# SIH-SUS: Heisenberg soft mixture of experts

Adaptation of the SIA-SUS MoE reference to admission-level mortality classification. Each fold uses all available prior rows, with six validation months and six subsequent Analysis months. The ten SIH-SUS outer test folds are **one month each**, unlike the SIA-SUS six-month test folds. The first fold therefore has only two months of inner training. These 2025 test months were previously inspected: results are exploratory.

Every sweep candidate trains its own classical and Heisenberg experts, selects their XGBoost parameters in validation, learns the router on Analysis, refits experts on all outer history and evaluates the frozen router on the test month. No test labels enter the selection or router training.

In [1]:
import os
if os.environ.get('PQFM_SIH_FIRST_FOLD_HILL') == '1':
    os.environ.setdefault('OMP_NUM_THREADS', '8')

import hashlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import minimize_scalar
from scipy.special import expit, logit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, auc, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, mean_squared_error,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

DATA_DIR = Path(os.environ.get('PQFM_SIH_DATA', '/home/pichau/pqfmlib-api/data/sih-sus/protocolo_comparacao'))
SEED = 42
N_JOBS = int(os.environ.get('PQFM_XGB_THREADS', '4'))
INNER_VALIDATION_MONTHS = 6
GAMMAS = (0.05, 0.1, 0.6)
TARGETS = {
    "classificacao": "morte",
}
ANALYSIS_MONTHS = 6
TEMPORAL_PROTOCOL = "temporal_holdout_validation6_analysis6_v1"

import joblib
from threadpoolctl import threadpool_limits


In [2]:
from contextlib import contextmanager
from contextvars import ContextVar
from datetime import datetime
from functools import wraps
from inspect import signature

_ACTIVE_TIMING = ContextVar("demanda_active_timing", default=None)
TIMING_COLUMNS = ["task", "fold", "stage_id", "parent_id", "depth", "stage",
                  "started_at", "finished_at", "seconds", "exclusive_seconds", "status"]


def format_duration(seconds):
    """Formata duração monotônica em HH:MM:SS.sss; não representa horário do dia."""
    hours, rest = divmod(float(seconds), 3600)
    minutes, rest = divmod(rest, 60)
    return f"{int(hours):02d}:{int(minutes):02d}:{rest:06.3f}"


def format_log_timestamp(timestamp):
    """Exibe horário local com dois espaços entre data e hora, sem sufixo de fuso."""
    return datetime.fromisoformat(timestamp).strftime("%Y-%m-%d  %H:%M:%S")


@contextmanager
def measure_stage(label):
    """Registra início/fim e tempo inclusivo/exclusivo de uma etapa do fold ativo.

    Usa perf_counter para duração e horário local com fuso apenas para os logs.
    parent_id/depth descrevem a hierarquia: não somar pais e filhos. Sem fold
    ativo, não instrumenta chamadas isoladas nem o benchmark CPU/GPU.
    """
    state = _ACTIVE_TIMING.get()
    if state is None:
        yield
        return
    stack, records = state["stack"], state["records"]
    row = dict(task=state["task"], fold=state["fold"], stage_id=len(records),
               parent_id=stack[-1] if stack else None, depth=len(stack), stage=label,
               started_at=datetime.now().astimezone().isoformat(timespec="seconds"),
               status="running")
    records.append(row)
    stack.append(row["stage_id"])
    prefix = f"{row['task']}: fold {row['fold']:02d} | " + "  " * row["depth"] + label
    print(f"[{format_log_timestamp(row['started_at'])}] {prefix}: início", flush=True)
    start = time.perf_counter()
    try:
        yield
    except BaseException:
        row["status"] = "failed"
        raise
    else:
        row["status"] = "completed"
    finally:
        row["seconds"] = time.perf_counter() - start
        row["finished_at"] = datetime.now().astimezone().isoformat(timespec="seconds")
        children = sum(r["seconds"] for r in records
                       if r["parent_id"] == row["stage_id"] and "seconds" in r)
        row["exclusive_seconds"] = max(0., row["seconds"] - children)
        stack.pop()
        status = "concluído" if row["status"] == "completed" else "interrompido/erro"
        print(f"[{format_log_timestamp(row['finished_at'])}] {prefix}: {status} em "
              f"{format_duration(row['seconds'])}", flush=True)


def timed_function(label):
    """Instrumenta funções sem mudar entradas/saídas; inclui kind/stage no rótulo."""
    def decorate(function):
        """Preserva assinatura/documentação da função e devolve seu invólucro de medição."""
        parameters = signature(function)
        @wraps(function)
        def wrapped(*args, **kwargs):
            """Encaminha argumentos e resultado sem alteração, registrando a duração no contexto ativo."""
            bound = parameters.bind(*args, **kwargs)
            bound.apply_defaults()
            detail = bound.arguments.get("kind", bound.arguments.get("stage"))
            name = f"{label} [{detail}]" if detail is not None else label
            with measure_stage(name):
                return function(*args, **kwargs)
        return wrapped
    return decorate


@contextmanager
def record_fold_timings(task, fold, label):
    """Cria uma coleta independente por fold e restaura o contexto mesmo em erro."""
    state = dict(task=task, fold=int(fold), records=[], stack=[])
    token = _ACTIVE_TIMING.set(state)
    try:
        with measure_stage(label):
            yield state["records"]
    finally:
        _ACTIVE_TIMING.reset(token)


## Data and temporal folds

In [3]:
FIRST_FOLD_DATA_LOAD = None
if os.environ.get('PQFM_SIH_FIRST_FOLD_HILL') == '1':
    dataset_start = time.perf_counter()
    dataset_io_before = Path('/proc/self/io').read_text()
source = DATA_DIR / 'base_modelagem_congelada.parquet'
data = pd.read_parquet(source)
raw_protocol = json.loads((DATA_DIR / 'protocolo.json').read_text())
data['mes'] = pd.to_datetime(data['mes'])
features = raw_protocol['features']
numeric = [c for c in features if pd.api.types.is_numeric_dtype(data[c])]
categorical = [c for c in features if c not in numeric]
for column in categorical:
    data[column] = data[column].astype(object).where(data[column].notna(), None)
# These are adapter columns only; the SIH unit remains the admission, not a CNES-month.
data['cnes'] = data['id_linha']
data['competencia'] = data['mes']
data['mes_alvo'] = data['mes']
protocol = {'features_numericas': numeric, 'features_categoricas': categorical}
GRID_CLASSIFICATION = {'learning_rate':[0.05,0.1,0.2], 'max_depth':[4,6,8], 'min_child_weight':[5,10,20]}
if 'PQFM_XGB_GRID' in os.environ:
    GRID_CLASSIFICATION = json.loads(os.environ['PQFM_XGB_GRID'])
with source.open('rb') as dataset_file:
    dataset_sha256 = hashlib.file_digest(dataset_file, 'sha256').hexdigest()
if os.environ.get('PQFM_SIH_FIRST_FOLD_HILL') == '1':
    FIRST_FOLD_DATA_LOAD = dict(seconds=time.perf_counter()-dataset_start,
                                io_before=dataset_io_before,
                                io_after=Path('/proc/self/io').read_text())

def split_outer_train(frame):
    months = pd.DatetimeIndex(sorted(frame.mes.unique()))
    if len(months) <= 12 or not months.equals(pd.date_range(months[0], months[-1], freq='MS')):
        raise ValueError('At least 13 consecutive history months are required for the 6+6 protocol.')
    validation_start, analysis_start = months[-12], months[-6]
    parts = (frame[frame.mes < validation_start], frame[(frame.mes >= validation_start) & (frame.mes < analysis_start)], frame[frame.mes >= analysis_start])
    if any(part.empty or part.morte.nunique() != 2 for part in parts):
        raise ValueError('Every temporal training role must contain both classes.')
    return parts

folds = []
for spec in raw_protocol['folds']:
    number, month = int(spec['fold']), pd.Timestamp(spec['teste_mes'])
    history = data[data.mes < month]
    test = data[data.fold_teste.eq(number).fillna(False)]
    inner, validation, analysis = split_outer_train(history)
    if test.empty or not test.mes.eq(month).all() or set(history.id_linha) & set(test.id_linha):
        raise ValueError(f'Fold {number}: invalid temporal test rows')
    folds.append({'fold':number, 'month':month, 'train':history.index, 'test':test.index,
                  'roles':[(str(x.mes.min().date()),str(x.mes.max().date()),len(x)) for x in (inner,validation,analysis)]})
assert len(folds) == 10
pd.DataFrame([{'fold':f['fold'],'test':f['month'],'inner / validation / analysis':f['roles']} for f in folds])

,fold,test,inner / validation / analysis
0,1,2025-03-01,"[(2024-01-01, 2024-02-01, 21876), (2024-03-01,..."
1,2,2025-04-01,"[(2024-01-01, 2024-03-01, 33803), (2024-04-01,..."
2,3,2025-05-01,"[(2024-01-01, 2024-04-01, 46221), (2024-05-01,..."
3,4,2025-06-01,"[(2024-01-01, 2024-05-01, 58295), (2024-06-01,..."
4,5,2025-07-01,"[(2024-01-01, 2024-06-01, 69986), (2024-07-01,..."
5,6,2025-08-01,"[(2024-01-01, 2024-07-01, 82298), (2024-08-01,..."
6,7,2025-09-01,"[(2024-01-01, 2024-08-01, 94422), (2024-09-01,..."
7,8,2025-10-01,"[(2024-01-01, 2024-09-01, 106062), (2024-10-01..."
8,9,2025-11-01,"[(2024-01-01, 2024-10-01, 118425), (2024-11-01..."
9,10,2025-12-01,"[(2024-01-01, 2024-11-01, 129885), (2024-12-01..."


## Classical expert, calibration, and the reference search grid

In [4]:
def make_preprocessor(protocol):
    """Constrói imputação/escala numérica e one-hot de UF; o chamador deve ajustar apenas no bloco de treino apropriado."""
    return ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median', keep_empty_features=True)), ('scaler', StandardScaler())]), protocol['features_numericas']), ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='__missing__', keep_empty_features=True)), ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), protocol['features_categoricas'])], remainder='drop')

def make_estimator(task, params):
    """Cria XGBoost de classificação com os parâmetros aprovados."""
    if task != "classificacao":
        raise ValueError("Este notebook executa somente classificação.")
    return XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", n_estimators=300,
        subsample=0.85, colsample_bytree=0.85, tree_method="hist",
        n_jobs=N_JOBS, random_state=SEED, verbosity=0, **params,
    )

def make_pipeline(task, params, protocol):
    """Encadeia pré-processamento e estimador não ajustados para impedir reutilização de estatísticas entre candidatos."""
    return Pipeline([('preprocessor', make_preprocessor(protocol)), ('model', make_estimator(task, params))])

@timed_function('Clássico: busca interna')
def tune_classic(data, fold, protocol, task, grid):
    """Ajusta candidatos somente em inner_train e seleciona por AP em inner_validation; retorna parâmetros e tabela, sem usar Analysis ou teste externo."""
    features = protocol['features_numericas'] + protocol['features_categoricas']
    target = TARGETS[task]
    train, validation = (data.iloc[fold[role]] for role in ('inner_train', 'inner_validation'))
    for block in (train, validation):
        if block[target].nunique() != 2:
            raise ValueError('Treino e validação de classificação exigem duas classes.')
    preprocessor = make_preprocessor(protocol).fit(train[features])
    train_features = preprocessor.transform(train[features])
    validation_features = preprocessor.transform(validation[features])
    rows = []
    for params in ParameterGrid(grid):
        start = time.perf_counter()
        model = make_estimator(task, params)
        model.fit(train_features, train[target])
        score = average_precision_score(validation[target], model.predict_proba(validation_features)[:, 1])
        if not np.isfinite(score):
            raise ValueError('Métrica interna não finita.')
        rows.append({'params': params, 'score': float(score), 'seconds': time.perf_counter() - start})
    if not rows:
        raise ValueError('Grid vazio.')
    select = max
    best = select(rows, key=lambda row: row['score'])
    return (dict(best['params']), pd.DataFrame(rows))


In [5]:
def fit_temperature(probability, y):
    """Minimiza NLL nos scores/rótulos da calibração; retorna temperatura positiva, sem alterar pesos do especialista."""
    probability = np.asarray(probability, dtype=float)
    y = np.asarray(y, dtype=int)
    if set(np.unique(y)) != {0, 1}:
        raise ValueError("Calibração exige ambas as classes.")
    logits = logit(np.clip(probability, 1e-7, 1 - 1e-7))

    def nll(log_temperature):
        """Calcula a perda de calibração para log-temperatura; retorna a média da log-loss numericamente estável."""
        scaled = logits / np.exp(log_temperature)
        return float(np.mean(np.logaddexp(0, scaled) - y * scaled))

    lower, upper = np.log(0.01), np.log(100.0)
    result = minimize_scalar(nll, bounds=(lower, upper), method="bounded")
    if not result.success:
        raise RuntimeError("Falha ao ajustar temperatura.")
    candidates = [0.0, lower, upper, float(result.x)]
    return float(np.exp(min(candidates, key=nll)))


def apply_temperature(probability, temperature):
    """Transforma probabilidades usando temperatura já ajustada; não aprende nem consulta rótulos."""
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError("Temperatura inválida.")
    return expit(logit(np.clip(probability, 1e-7, 1 - 1e-7)) / temperature)


def best_f1_threshold(y, score):
    """Seleciona o limiar de maior F1 no bloco fornecido de calibração; retorna um score observado, nunca um limiar escolhido no teste."""
    precision, recall, thresholds = precision_recall_curve(y, score)
    if not len(thresholds):
        raise ValueError("Não foi possível determinar um limiar.")
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    return float(thresholds[int(np.argmax(f1))])


def classification_metrics(y, score, prediction):
    """Recebe rótulos, scores e classes previstos; retorna métricas e matriz de confusão, sem ajustar modelos."""
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    both = len(np.unique(y)) == 2
    precision, recall, _ = precision_recall_curve(y, score)
    return {
        "AP": average_precision_score(y, score) if both else np.nan,
        "AUPRC_trapezoidal": auc(recall, precision) if both else np.nan,
        "ROC-AUC": roc_auc_score(y, score) if both else np.nan,
        "Accuracy": accuracy_score(y, prediction),
        "BalancedAccuracy": balanced_accuracy_score(y, prediction),
        "Precision": precision_score(y, prediction, zero_division=0),
        "Recall": recall_score(y, prediction, zero_division=0),
        "F1": f1_score(y, prediction, zero_division=0),
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


In [6]:
@timed_function('Clássico: ajuste e calibração')
def fit_classic(expert, inner_train, inner_validation, calibration,
                protocol, task, grid):
    # As tabelas recebidas não contêm o teste nem o bloco do roteador.
    """Busca interna e refit em expert; calibration=None desativa ajuste local para usar posteriormente a calibração na validação interna."""
    tuning_data = pd.concat([inner_train, inner_validation], ignore_index=True)
    inner_fold = {
        "inner_train": np.arange(len(inner_train)),
        "inner_validation": np.arange(len(inner_train), len(tuning_data)),
    }
    start = time.perf_counter()
    params, tuning = tune_classic(tuning_data, inner_fold, protocol, task, grid)
    features = protocol["features_numericas"] + protocol["features_categoricas"]
    model = make_pipeline(task, params, protocol)
    with measure_stage('Clássico: reajuste final em expert'):
        model.fit(expert[features], expert[TARGETS[task]])
    fitted = {"pipeline": model, "params": params, "tuning": tuning, "task": task}
    if task == "classificacao" and calibration is not None:
        with measure_stage('Clássico: calibração (temperatura e F1)'):
            probability = model.predict_proba(calibration[features])[:, 1]
            y = calibration[TARGETS[task]]
            temperature = fit_temperature(probability, y)
            score = apply_temperature(probability, temperature)
            fitted.update(temperature=temperature, threshold=best_f1_threshold(y, score))
    fitted["fit_seconds"] = time.perf_counter() - start
    return fitted


In [7]:
import copy
import torch
import importlib.util
import sys
if importlib.util.find_spec('triton') is not None:
    if 'cudaq' in sys.modules and 'triton' not in sys.modules:
        raise RuntimeError('CUDA-Q foi carregado antes do Triton. Reinicie o kernel e execute as definições em ordem antes do benchmark ou dos experimentos.')
    import triton
from sklearn.decomposition import PCA
torch.set_num_threads(1)

def validate_task_data(X, y, task):
    """Confere matriz X e alvo y finitos, alinhados e compatíveis com a tarefa; retorna arrays validados ou gera erro."""
    X, y = (np.asarray(X, dtype=float), np.asarray(y, dtype=float))
    if task not in TARGETS or X.ndim != 2 or y.ndim != 1 or (len(X) != len(y)) or (not len(y)) or (not np.isfinite(X).all()) or (not np.isfinite(y).all()):
        raise ValueError('Tarefa, features ou alvos inválidos.')
    if task == 'classificacao' and set(np.unique(y)) != {0, 1}:
        raise ValueError('O treinamento do classificador exige ambas as classes.')
    return (X, y)


In [8]:
def heisenberg_execution_config(device, expectation_method, mps=False, framework="qiskit", precision="double"):
    """Valida framework, expectativas e dispositivo sem fallback entre simuladores."""
    if framework not in ("qiskit", "cudaq", "tilelang"):
        raise ValueError("HEISENBERG_FRAMEWORK deve ser qiskit, cudaq ou tilelang.")
    if precision not in ("single", "double"):
        raise ValueError("HEISENBERG_PRECISION deve ser single ou double.")
    if framework == "qiskit" and mps and precision == "single":
        raise ValueError("Qiskit MPS não suporta precisão single; use double.")
    if device not in ("CPU", "GPU"):
        raise ValueError("HEISENBERG_DEVICE deve ser CPU ou GPU.")
    methods = ("exact",) if framework == "tilelang" else (("statevector", "shots", "exact") if framework == "cudaq" else ("statevector", "shots"))
    if expectation_method not in methods:
        raise ValueError(f"expectation_method para {framework} deve ser um de {methods}.")
    if not isinstance(mps, bool):
        raise ValueError("HEISENBERG_MPS deve ser booleano.")
    if framework == "tilelang" and (mps or precision != "double"):
        raise ValueError("TileLang exige statevector (mps=False) e precisão double.")
    if mps and expectation_method == "statevector":
        raise ValueError("Expectativas exatas com MPS exigem framework='cudaq' e expectation_method='exact'.")
    if framework in ("cudaq", "tilelang") and device != "GPU":
        raise ValueError(f"{framework} requer HEISENBERG_DEVICE='GPU'.")
    if framework == "qiskit" and mps and device != "CPU":
        print("MPS do Qiskit permite apenas CPU; HEISENBERG_DEVICE não se aplica neste modo.", flush=True)
    return dict(
        framework=framework, precision=precision, expectation_method=expectation_method, mps=mps,
        use_gpu_statevector=not mps and device == "GPU",
        statevector_device=None if mps and framework == "qiskit" else device,
    )


In [9]:
# ÚNICA CÉLULA A EDITAR para configurar o Heisenberg.
HEISENBERG_FRAMEWORK = "tilelang"  # "qiskit", "cudaq" ou "tilelang".
HEISENBERG_PRECISION = "double"  # "single" (FP32) ou "double" (FP64); Qiskit MPS e TileLang exigem double.
HEISENBERG_DEVICE = "GPU"  # "CPU" ou "GPU"; não há fallback automático.
HEISENBERG_EXPECTATION_METHOD = "exact"  # TileLang: "exact"; CUDA-Q: "exact" ou "shots"; Qiskit: "statevector" ou "shots".
HEISENBERG_MPS = False  # True: MPS (Qiskit/CPU ou CUDA-Q/GPU); False: statevector.

HEISENBERG_CONFIG = dict(
    n_components=54, q_enc=19,  # PCA: 75 componentes (val-selected Phase3); mapa completa zeros.
    topology="chain", interaction_axes=("x", "y", "z"), encoding_mode="multi_axis", #shared_feature
    hybrid_shap_top_k=50,  # Top features por média de |SHAP| (val-selected Phase3).
    R=2, alpha=0.1,  # Val-selected Phase4 (val-mean 0.6466).
    use_tanh_scaling=True, measure_2local_diagonal=False, measure_cross_observables=False,  # 3 observáveis locais/qubit.
    shots=4096,  # Usado SOMENTE no modo "shots"; ignorado nas expectativas exatas.
    ideal=True, simulation=True, fakebackend=False,  # Apenas simulação ideal.
    mps_max_bond_dimension=16, mps_truncation_threshold=0.0,  # None: teto automático no CUDA-Q; inteiro: limite explícito. Cutoff zero usa piso numérico no CUDA-Q.
    **heisenberg_execution_config(HEISENBERG_DEVICE, HEISENBERG_EXPECTATION_METHOD, HEISENBERG_MPS, HEISENBERG_FRAMEWORK, HEISENBERG_PRECISION),
)
effective_device = HEISENBERG_CONFIG["statevector_device"] or "CPU"
print("Heisenberg configurado:", HEISENBERG_FRAMEWORK, HEISENBERG_EXPECTATION_METHOD, HEISENBERG_PRECISION,
      f"{'MPS' if HEISENBERG_MPS else 'statevector'}/{effective_device}; {HEISENBERG_CONFIG['q_enc']} qubits")
# O backend efetivo será conferido e exibido ao ajustar cada mapa.


Heisenberg configurado: tilelang exact double statevector/GPU; 19 qubits
Heisenberg configurado: tilelang exact double statevector/GPU; 19 qubits


In [10]:
def make_heisenberg_map(config, seed=SEED):
    """Valida opções de simulação ideal e disponibilidade do dispositivo; retorna mapa público pqfmlib sem fallback nem hardware real."""
    if not config['ideal'] or not config['simulation'] or config['fakebackend'] or config['measure_2local_diagonal'] or config.get('measure_cross_observables', False):
        raise ValueError('Heisenberg requer simulação ideal e observáveis locais.')
    method = config['expectation_method']
    framework = config.get('framework', 'qiskit')
    device = 'CPU' if config['mps'] and framework == 'qiskit' else config['statevector_device']
    expected = heisenberg_execution_config(device, method, config['mps'], framework, config.get('precision', 'double'))
    if any((config.get(key, 'qiskit' if key == 'framework' else ('double' if key == 'precision' else None)) != value for key, value in expected.items())):
        raise ValueError('Opções contraditórias: use a célula exclusiva de configuração Heisenberg.')
    # if config.get('topology', 'chain') != 'chain':
    #     raise ValueError('Este experimento usa a topologia chain.')
    features_per_edge = len(config['interaction_axes']) if config.get('encoding_mode') == 'multi_axis' else 1
    block_capacity = (config['q_enc'] - 1) * features_per_edge
    if block_capacity < 1 or config['n_components'] < 1 or config['shots'] < 1:
        raise ValueError('O mapa requer capacidade, componentes e shots positivos.')
    if framework == 'qiskit':
        from qiskit_aer import AerSimulator
        if device not in AerSimulator().available_devices():
            raise RuntimeError(f'Aer não disponibiliza {device}. Verifique instalação/driver; sem fallback.')
    from pqfmlib import HeisenbergProjectiveQFM
    return HeisenbergProjectiveQFM(name_file='demanda_pressao_heisenberg', seed=seed, save_circuit_drawings=False, **{key: value for key, value in config.items() if key not in {'n_components', 'hybrid_shap_top_k', 'hybrid_pca_variance'}})

def orient_pca_components(components):
    """Use the first nearly maximal loading to break numerical sign ties."""
    magnitude = np.abs(components)
    anchors = np.argmax(magnitude >= magnitude.max(axis=1, keepdims=True) - 1e-9, axis=1)
    components *= np.sign(components[np.arange(len(components)), anchors])[:, None]


class OneHotCovariancePCA:
    """Covariance PCA with dense numeric and sparse one-hot cross-products."""

    def __init__(self, components, categorical_start):
        self.components = components
        self.categorical_start = categorical_start

    def fit(self, X):
        from scipy.sparse import csr_matrix
        from sklearn.utils.extmath import svd_flip

        n, self.n_features_in_ = X.shape
        numeric = np.ascontiguousarray(X[:, :self.categorical_start])
        categorical = csr_matrix(X[:, self.categorical_start:])
        self.mean_ = np.concatenate([numeric.mean(axis=0), np.asarray(categorical.mean(axis=0)).ravel()])
        cross = (categorical.T @ numeric).T
        gram = np.block([[numeric.T @ numeric, cross],
                         [cross.T, (categorical.T @ categorical).toarray()]])
        gram -= n * np.outer(self.mean_, self.mean_)
        gram /= n - 1
        values, vectors = np.linalg.eigh(gram)
        order = slice(None, -self.components - 1, -1)
        _, components = svd_flip(None, vectors[:, order].T.copy(), u_based_decision=False)
        self.components_ = components
        self.explained_variance_ = values[order].clip(min=0)
        return self

    def transform(self, X):
        return X @ self.components_.T - self.mean_ @ self.components_.T


class HeisenbergFeatures:
    """PCA e mapa congelados; registra operações sem salvar artefatos da biblioteca."""

    def __init__(self, config=None, seed=SEED, factory=make_heisenberg_map):
        """Inicializa HeisenbergFeatures com a configuração recebida; treinamento ocorre somente em fit ou na otimização explícita."""
        self.config = dict(HEISENBERG_CONFIG, **config or {})
        self.seed, self.factory = (seed, factory)
        self.calls = []

    @timed_function('Heisenberg: preparar PCA/mapa')
    def fit(self, X, stage='expert', *, pca_threads=4, structure_only=False, categorical_start=None, stable_pca_sign=False):
        """Ajusta PCA/mapa em X do papel indicado e valida backend; retorna self. Não extrai todas as features: isso cabe a transform."""
        X = np.asarray(X, dtype=float)
        requested = self.config['n_components']
        if X.ndim != 2 or X.shape[0] < 2 or X.shape[1] < 1 or not np.isfinite(X).all():
            raise ValueError('PCA requer matriz finita com pelo menos duas amostras e uma feature.')
        if isinstance(requested, bool) or not isinstance(requested, (int, np.integer)) or requested < 1:
            raise ValueError('n_components deve ser um inteiro positivo.')
        self.n_components_requested_ = int(requested)
        self.n_components_ = min(self.n_components_requested_, X.shape[1], X.shape[0] - 1)
        self.n_zero_slots_ = self.n_components_requested_ - self.n_components_
        print(f'PCA Heisenberg [{stage}]: {self.n_components_}/{requested} componentes; '
              f'{self.n_zero_slots_} posições de encoding preenchidas com zero pelo mapa.', flush=True)
        start = time.perf_counter()
        with threadpool_limits(limits=pca_threads, user_api='blas'):
            pca = (PCA(n_components=self.n_components_, svd_solver='covariance_eigh')
                   if categorical_start is None else OneHotCovariancePCA(self.n_components_, categorical_start))
            self.pca_ = pca.fit(X)
        if stable_pca_sign:
            orient_pca_components(self.pca_.components_)
        rng = np.random.get_state()
        try:
            self.map_ = self.factory(self.config, self.seed)
            if structure_only:
                from pqfmlib import HeisenbergProjectiveQFM
                if type(self.map_) is not HeisenbergProjectiveQFM:
                    raise TypeError('Structure-only fit requires the reference Heisenberg map.')
                # This map's fit inspects only input width, then initializes its
                # angle structure from zeros. PCA itself is fitted to all X.
                self.map_.fit(np.zeros((1, self.n_components_), dtype=float))
            else:
                self.map_.fit(self.pca_.transform(X))
        finally:
            np.random.set_state(rng)
        framework = self.config.get('framework', 'qiskit')
        mps = self.config['mps']
        if framework == 'tilelang':
            backend = self.map_.backend
            if backend.precision != self.config['precision'] or backend.device != 'GPU':
                raise RuntimeError('Backend TileLang divergente.')
            self.backend_ = dict(framework=framework, method=backend.target, device=backend.device, precision=backend.precision)
        elif framework == 'cudaq':
            backend = self.map_.backend
            target = 'tensornet-mps' if mps else 'nvidia'
            if backend.precision != self.config.get('precision', 'double') or backend.target != target or backend.mps != mps or backend.mps_max_bond_dimension != self.config['mps_max_bond_dimension'] or (backend.mps_truncation_threshold != self.config['mps_truncation_threshold']):
                raise RuntimeError(f'Backend Heisenberg divergente: target={backend.target}')
            self.backend_ = dict(framework=framework, method=backend.target, device='GPU', precision=backend.precision, bond_dimension=backend.mps_max_bond_dimension, truncation=backend.mps_truncation_threshold)
        else:
            options = self.map_.backend.options
            self.backend_ = {'framework': framework, 'method': str(options.method), 'device': str(options.device), 'precision': str(options.precision), 'bond_dimension': options.matrix_product_state_max_bond_dimension, 'truncation': float(options.matrix_product_state_truncation_threshold)}
            expected = dict(method='matrix_product_state' if mps else 'statevector', device='CPU' if mps else self.config['statevector_device'], precision=self.config.get('precision', 'double'))
            if mps:
                expected.update(
                    bond_dimension=self.config['mps_max_bond_dimension'],
                    truncation=self.config['mps_truncation_threshold'],
                )
            if any((self.backend_[key] != value for key, value in expected.items())):
                raise RuntimeError(f'Backend Heisenberg divergente: {self.backend_}')
        print(f'Heisenberg [{stage}]: {self.backend_['framework']}/{self.backend_['method']}/{self.backend_['device']}; expectativas={self.config['expectation_method']}; {self.config['q_enc']} qubits; mapa preparado, extração ainda não iniciada', flush=True)
        self.n_fit_samples_ = len(X)
        self.calls.append(dict(stage=stage, operation='fit', n_rows=len(X),
                               n_components_requested=self.n_components_requested_,
                               n_components_effective=self.n_components_, zero_encoding_slots=self.n_zero_slots_,
                               seconds=time.perf_counter() - start))
        return self

    @timed_function('Heisenberg: extração')
    def transform(self, X, stage='inference'):
        """Aplica PCA/mapa congelados a X; retorna 3*q_enc features por linha e registra tempo/papel, sem reajuste."""
        X = np.asarray(X, dtype=float)
        if X.ndim != 2 or X.shape[1] != self.pca_.n_features_in_ or (not np.isfinite(X).all()):
            raise ValueError('Features inválidas para Heisenberg.')
        if not len(X):
            return np.empty((0, 3 * self.config['q_enc']))
        start = time.perf_counter()
        quantum = np.asarray(self.map_.transform(self.pca_.transform(X)), dtype=float)
        if quantum.shape != (len(X), 3 * self.config['q_enc']) or not np.isfinite(quantum).all():
            raise RuntimeError('Saída quântica com dimensão ou valores inválidos.')
        self.calls.append(dict(stage=stage, operation='transform', n_rows=len(X), seconds=time.perf_counter() - start))
        return quantum

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.utils.validation import check_is_fitted
import shap


class HybridSHAPSelector(TransformerMixin, BaseEstimator):
    """Seleciona colunas originais por média de |TreeSHAP|, só no treino recebido."""

    def __init__(self, estimator, top_k=50):
        self.estimator = estimator
        self.top_k = top_k

    def fit(self, X, y, *, training_matrix=None):
        X = np.asarray(X, dtype=float)
        if (isinstance(self.top_k, bool) or not isinstance(self.top_k, (int, np.integer))
                or self.top_k < 1):
            raise ValueError("top_k deve ser um inteiro positivo.")
        if X.ndim != 2 or not len(X) or not X.shape[1] or not np.isfinite(X).all():
            raise ValueError("SHAP requer uma matriz de treino finita e não vazia.")
        self.n_features_in_, self.n_samples_seen_ = X.shape[1], len(X)
        # Mesmo XGBoost/parâmetros do preditor, ajustado antes da seleção.
        if training_matrix is None:
            ranking_model = clone(self.estimator).fit(X, y)
        else:
            ranking_model = _fit_prepared_xgboost(clone(self.estimator), training_matrix)
        explainer = shap.TreeExplainer(ranking_model, feature_perturbation="tree_path_dependent",
                                       model_output="raw")
        total = np.zeros(X.shape[1], dtype=float)
        # Todas as linhas de treino; lotes limitam memória sem amostragem.
        for start in range(0, len(X), 1024):
            batch = X[start:start + 1024]
            values = np.asarray(explainer.shap_values(batch), dtype=float)
            if values.shape != batch.shape or not np.isfinite(values).all():
                raise ValueError("TreeSHAP binário retornou valores inválidos.")
            total += np.abs(values).sum(axis=0)
        self.feature_importances_ = total / len(X)
        # Empates são resolvidos pelo índice original, de forma determinística.
        self.ranking_ = np.argsort(-self.feature_importances_, kind="stable")
        self.n_selected_ = min(self.top_k, X.shape[1])
        self.selected_indices_ = self.ranking_[:self.n_selected_]
        return self

    def transform(self, X):
        check_is_fitted(self, "selected_indices_")
        X = np.asarray(X, dtype=float)
        if X.ndim != 2 or X.shape[1] != self.n_features_in_ or not np.isfinite(X).all():
            raise ValueError("Features incompatíveis com a seleção SHAP ajustada.")
        return X[:, self.selected_indices_]


def quantum_estimator(task, params, config=None):
    """Ajusta ranking SHAP e retreina o XGBoost nas top features, somente no treino."""
    cfg = HEISENBERG_CONFIG if config is None else config
    estimator = make_estimator(task, params)
    return Pipeline([
        ('hybrid_scaler', StandardScaler()),
        ('hybrid_shap', HybridSHAPSelector(estimator, top_k=cfg['hybrid_shap_top_k'])),
        ('model', clone(estimator)),
    ])


def hybrid_selection_diagnostics(expert):
    """Identifica cada coluna concatenada e seu ranking no modelo já ajustado."""
    selector = expert['model'].model.named_steps['hybrid_shap']
    quantum_names = [f'quantum_{axis}_{i}' for i in range(expert['config']['q_enc'])
                     for axis in ('z', 'x', 'y')]
    classic_names = list(expert['preprocessor'].get_feature_names_out())
    names = quantum_names + classic_names
    if len(names) != selector.n_features_in_:
        raise ValueError("Nomes das features divergentes da matriz híbrida.")
    selected = set(selector.selected_indices_.tolist())
    ranking = [dict(rank=rank, index=int(i), feature=names[i],
                    origin='quantum' if i < len(quantum_names) else 'classic',
                    mean_abs_shap=float(selector.feature_importances_[i]), selected=i in selected)
               for rank, i in enumerate(selector.ranking_, start=1)]
    quantum_count = sum(i < len(quantum_names) for i in selected)
    return dict(method='tree_shap_mean_absolute_train', top_k=selector.top_k,
                n_input=selector.n_features_in_, n_selected=selector.n_selected_,
                n_train=selector.n_samples_seen_, n_quantum_selected=quantum_count,
                n_classic_selected=selector.n_selected_ - quantum_count, ranking=ranking)

class HeisenbergExpert:

    def __init__(self, task, feature_map, model):
        """Inicializa HeisenbergExpert com a configuração recebida; treinamento ocorre somente em fit ou na otimização explícita."""
        self.task, self.feature_map, self.model = (task, feature_map, model)

    def predict_raw(self, X, stage='inference'):
        """Extrai features com mapa congelado e aplica estimador treinado; retorna probabilidades ou demanda não negativa."""
        if not len(X):
            return np.empty(0)
        quantum = self.feature_map.transform(X, stage=stage)
        hybrid = np.column_stack([quantum, X])
        return self.model.predict_proba(hybrid)[:, 1]

@timed_function('Heisenberg: seleção interna')
def tune_heisenberg(inner_train, inner_validation, protocol, task, grid, config=None, factory=make_heisenberg_map):
    """Ajusta pré-processamento/PCA/mapa em inner_train, extrai treino/validação uma vez e reutiliza no grid; retorna melhores parâmetros, tabela e chamadas."""
    features = protocol['features_numericas'] + protocol['features_categoricas']
    target = TARGETS[task]
    preprocessing = make_preprocessor(protocol).fit(inner_train[features])
    Xt = preprocessing.transform(inner_train[features])
    Xv = preprocessing.transform(inner_validation[features])
    validate_task_data(Xt, inner_train[target], task)
    if task == 'classificacao' and inner_validation[target].nunique() != 2:
        raise ValueError('Validação de classificação exige ambas as classes.')
    feature_map = HeisenbergFeatures(config, factory=factory).fit(Xt, stage='inner_train')
    qt = np.column_stack([feature_map.transform(Xt, stage='inner_train'), Xt])
    qv = np.column_stack([feature_map.transform(Xv, stage='inner_validation'), Xv])
    rows = []
    with measure_stage('Heisenberg: grid ML sobre features reutilizadas'):
        for params in ParameterGrid(grid):
            start = time.perf_counter()
            model = quantum_estimator(task, params, feature_map.config).fit(qt, inner_train[target])
            score = average_precision_score(inner_validation[target], model.predict_proba(qv)[:, 1])
            if not np.isfinite(score):
                raise ValueError('Métrica interna quântica não finita.')
            rows.append(dict(params=params, score=float(score), seconds=time.perf_counter() - start))
    if not rows:
        raise ValueError('Grid quântico vazio.')
    select = max
    best = dict(select(rows, key=lambda row: row['score'])['params'])
    return (best, pd.DataFrame(rows), feature_map.calls)


## Soft router from the reference

In [11]:
# Roteador do MoE Heisenberg de classificação; grid independente dos especialistas.
ROUTER_LEARNING_RATE_GRID = [0.01, 0.03, 0.05, 0.1, 0.2]
ROUTER_DEFAULT_LEARNING_RATE = 0.1  # Só quando o treino interno de z tem uma classe.
ROUTER_PARAMS = dict(
    objective="binary:logistic", eval_metric="logloss", tree_method="hist",
    n_estimators=100, max_depth=3, subsample=1.0, colsample_bytree=1.0,
    random_state=SEED, n_jobs=N_JOBS, verbosity=0,
)
ROUTER_PROTOCOL = "xgboost_analysis6_last_month_logloss_v1"


In [12]:
def expert_output(fitted, frame, protocol, stage='inference', uncalibrated=False):
    """Aplica o especialista congelado ao papel informado; retorna scores e decisões calibradas, ou demanda, sem treinamento."""
    features = protocol['features_numericas'] + protocol['features_categoricas']
    if fitted['kind'] == 'classic':
        model = fitted['pipeline']
        raw = model.predict_proba(frame[features])[:, 1]
    else:
        values = fitted['preprocessor'].transform(frame[features])
        raw = fitted['model'].predict_raw(values, stage=stage)
    raw = np.asarray(raw, dtype=float)
    if raw.shape != (len(frame),) or not np.isfinite(raw).all():
        raise ValueError('Saída inválida do especialista.')
    if ((raw < 0) | (raw > 1)).any():
        raise ValueError('Probabilidades fora de [0,1].')
    if uncalibrated:
        return (raw, (raw >= 0.5).astype(int))
    score = apply_temperature(raw, fitted.get('temperature', 1.0))
    return (score, (score >= fitted.get('threshold', 0.5)).astype(int))

def router_targets(task, y, classic_prediction, quantum_prediction):
    """Constrói z nas previsões de Analysis: secundário acerta e clássico erra; empates favorecem o clássico."""
    y = np.asarray(y)
    classic_prediction, quantum_prediction = map(np.asarray, (classic_prediction, quantum_prediction))
    if any((v.shape != y.shape or not np.isfinite(v).all() for v in (y, classic_prediction, quantum_prediction))):
        raise ValueError('Vetores inválidos para construir o alvo do roteador.')
    return ((quantum_prediction == y) & (classic_prediction != y)).astype(int)

class ConstantRouter:
    """Decisão determinística quando o bloco de análise tem uma única preferência."""

    def __init__(self, value):
        """Guarda a decisão constante quando Analysis contém uma única classe z; não requer fit."""
        self.value = int(value)

    def predict_proba(self, X):
        """Retorna probabilidades determinísticas 0/1 quando z tem uma classe; mantém interface binária sem inventar exemplos."""
        p = np.full(len(X), self.value, dtype=float)
        return np.column_stack([1 - p, p])

def make_router(learning_rate):
    """Cria XGBoost binário para z; não altera os estimadores dos especialistas."""
    return XGBClassifier(learning_rate=learning_rate, **ROUTER_PARAMS)

def fit_router(X, z, months, learning_rates=None, *, selection_X=None, selection_z=None, search_workers=1, candidate_factory=make_router, prepared_fit=None):
    """Seleciona learning_rate dentro de Analysis e retorna router + diagnóstico.

    X/z são features/alvos definitivos do refit; selection_X/selection_z são
    dados auxiliares sem influência do último mês nos ajustes preparatórios;
    months são meses-alvo alinhados por linha. O último mês valida os candidatos
    treinados nos meses anteriores (todos os meses OOF anteriores ao último). Minimiza log-loss
    binária, inclusive se a validação só tem uma classe; nunca usa o teste externo.
    Depois seleciona um estimador novo e reajusta em todo o conjunto OOF.
    Uma classe no Analysis inteiro gera ConstantRouter. Uma classe apenas no
    treino interno impede tuning: refit em todo Analysis com taxa fixa explícita.
    """
    from sklearn.metrics import log_loss
    X, z = (np.asarray(X, dtype=float), np.asarray(z))
    months = pd.DatetimeIndex(pd.to_datetime(months))
    if X.ndim != 2 or z.ndim != 1 or (not len(z)) or (len(X) != len(z)) or (len(months) != len(z)) or (not np.isfinite(X).all()) or (not np.isin(z, [0, 1]).all()) or months.isna().any():
        raise ValueError('Analysis exige X finito, z binário e meses alinhados.')
    z = z.astype(int)
    periods = months.to_period('M')
    unique = periods.unique().sort_values()
    if len(unique) < 2 or not unique.equals(pd.period_range(unique[0], unique[-1], freq='M')):
        raise ValueError('Analysis deve conter os meses consecutivos configurados, pelo menos dois.')
    rates = list(ROUTER_LEARNING_RATE_GRID if learning_rates is None else learning_rates)
    if not rates or any((isinstance(rate, bool) or not np.isfinite(rate) or rate <= 0 for rate in rates)) or len(set(rates)) != len(rates):
        raise ValueError('Grid do router deve conter learning_rates positivos, finitos e distintos.')
    fallback = float(ROUTER_DEFAULT_LEARNING_RATE)
    if not np.isfinite(fallback) or fallback <= 0:
        raise ValueError('Taxa fixa de fallback do router inválida.')
    selection_X = X if selection_X is None else np.asarray(selection_X, dtype=float)
    selection_z = z if selection_z is None else np.asarray(selection_z)
    if selection_X.ndim != 2 or len(selection_X) != len(z) or selection_z.shape != z.shape or (not np.isfinite(selection_X).all()) or (not np.isin(selection_z, [0, 1]).all()):
        raise ValueError('Dados auxiliares do router inválidos.')
    selection_z = selection_z.astype(int)
    train = np.asarray(periods < unique[-1])
    validation = np.asarray(periods == unique[-1])
    diagnostic = dict(protocol=ROUTER_PROTOCOL, selection_metric='binary_log_loss', selection_direction='min', learning_rate_grid=list(rates), selected_learning_rate=None, default_learning_rate=fallback, tie_break='smallest_learning_rate', inner_train_months=[str(month) for month in unique[:-1]], inner_validation_months=[str(unique[-1])], inner_train_n=int(train.sum()), inner_validation_n=int(validation.sum()), inner_train_class_counts=np.bincount(selection_z[train], minlength=2).tolist(), inner_validation_class_counts=np.bincount(selection_z[validation], minlength=2).tolist(), analysis_class_counts=np.bincount(z, minlength=2).tolist(), refit_n=len(z), tuning=[])
    if np.unique(z).size == 1:
        diagnostic.update(status='constant_analysis', reason='Uma classe em todo o Analysis.', estimator='ConstantRouter', constant=int(z[0]))
        print(f'Router: {diagnostic['reason']} Decisão constante={int(z[0])}.', flush=True)
        return (ConstantRouter(z[0]), diagnostic)
    if np.unique(selection_z[train]).size == 1:
        chosen = fallback
        diagnostic.update(status='untuned_single_class_inner_train', reason='Treino interno com uma classe: grid não executado; taxa fixa no refit.')
        print(f'Router: {diagnostic['reason']} learning_rate={chosen:g}.', flush=True)
    else:
        with measure_stage('Router: grid learning_rate (último mês de Analysis)'):
            train_X, train_y = selection_X[train], selection_z[train]
            validation_X = selection_X[validation]

            def score_rate(rate):
                start = time.perf_counter()
                candidate = candidate_factory(rate)
                candidate = (candidate.fit(train_X, train_y) if prepared_fit is None
                             else prepared_fit(candidate, train_y, 'selection'))
                probability = candidate.predict_proba(validation_X)[:, 1]
                if probability.shape != (int(validation.sum()),) or not np.isfinite(probability).all() or ((probability < 0) | (probability > 1)).any():
                    raise ValueError('Probabilidades inválidas na validação do router.')
                loss = float(log_loss(selection_z[validation], probability, labels=[0, 1]))
                return dict(learning_rate=float(rate), log_loss=loss, seconds=time.perf_counter() - start)
            if search_workers == 1:
                diagnostic['tuning'] = [score_rate(rate) for rate in rates]
            else:
                from concurrent.futures import ThreadPoolExecutor
                with ThreadPoolExecutor(max_workers=min(search_workers, len(rates))) as pool:
                    diagnostic['tuning'] = list(pool.map(score_rate, rates))
            chosen = min(diagnostic['tuning'], key=lambda row: (row['log_loss'], row['learning_rate']))['learning_rate']
        diagnostic.update(status='tuned', reason='Seleção no último mês; refit em todo Analysis.')
    with measure_stage('Router: refit em todos os meses de Analysis'):
        gate = candidate_factory(chosen)
        gate = gate.fit(X, z) if prepared_fit is None else prepared_fit(gate, z, 'refit')
    diagnostic.update(selected_learning_rate=float(chosen), estimator='XGBClassifier', effective_params=gate.get_params(deep=False))
    print(f'Router: {diagnostic['status']}; learning_rate={chosen:g}; refit={len(z)} amostras.', flush=True)
    return (gate, diagnostic)

@timed_function('Secundário: ajuste e calibração')
def fit_secondary(kind, primary, expert, inner_train, inner_validation, calibration, protocol, task, grid, config=None, factory=make_heisenberg_map):
    """Busca/refit Heisenberg; calibration=None mantém saída bruta para a calibração na validação interna posterior."""
    start = time.perf_counter()
    features = protocol['features_numericas'] + protocol['features_categoricas']
    preprocessing = primary['pipeline'].named_steps['preprocessor']
    X = preprocessing.transform(expert[features])
    y = expert[TARGETS[task]].to_numpy()
    fitted = {'kind': kind, 'task': task, 'preprocessor': preprocessing}
    if kind == 'Heisenberg':
        params, tuning, inner_calls = tune_heisenberg(inner_train, inner_validation, protocol, task, grid, config, factory)
        feature_map = HeisenbergFeatures(config, factory=factory).fit(X, stage='expert')
        quantum = feature_map.transform(X, stage='expert')
        with measure_stage('Heisenberg: reajuste final do estimador ML'):
            model = quantum_estimator(task, params, feature_map.config).fit(np.column_stack([quantum, X]), y)
        fitted.update(model=HeisenbergExpert(task, feature_map, model), params=params, tuning=tuning, inner_calls=inner_calls, config=feature_map.config)
    else:
        raise ValueError(f'Especialista desconhecido: {kind}')
    if task == 'classificacao' and calibration is not None:
        with measure_stage('Secundário: calibração (extração, temperatura e F1)'):
            raw = fitted['model'].predict_raw(preprocessing.transform(calibration[features]), stage='calibration')
            temperature = fit_temperature(raw, calibration[TARGETS[task]])
            score = apply_temperature(raw, temperature)
            fitted.update(temperature=temperature, threshold=best_f1_threshold(calibration[TARGETS[task]], score))
    fitted['fit_seconds'] = time.perf_counter() - start
    return fitted


In [13]:
def validation_calibration(raw, y):
    """Temperatura primeiro, threshold F1 depois, somente na validação interna."""
    temperature = fit_temperature(raw, y)
    return dict(temperature=temperature,
                threshold=best_f1_threshold(y, apply_temperature(raw, temperature)))


def calibrated_prediction(raw, calibration):
    score = apply_temperature(raw, calibration["temperature"])
    return score, (score >= calibration["threshold"]).astype(int)


def initial_expert_diagnostics(experts, inner):
    details = {}
    for name, expert in experts.items():
        details[name] = dict(train_n=len(inner), train_end=str(inner.mes_alvo.max()),
                             fit_seconds=expert["fit_seconds"],
                             temperature=expert["temperature"], threshold=expert["threshold"])
        if "params" in expert:
            details[name].update(params=copy.deepcopy(expert["params"]),
                                 tuning=expert["tuning"].to_dict("records"))
        if name == "Heisenberg":
            details[name]["hybrid_selection"] = hybrid_selection_diagnostics(expert)
            details[name].update(inner_calls=copy.deepcopy(expert["inner_calls"]),
                                 initial_calls=copy.deepcopy(expert["model"].feature_map.calls))
    return details


@timed_function('Experts: refit final sem nova busca')
def refit_experts(outer_train, protocol, task, initial, *, factory=make_heisenberg_map):
    """Reajusta modelos/PCA no outer train; transfere parâmetros, temperatura e threshold."""
    features, target = protocol["features_numericas"] + protocol["features_categoricas"], TARGETS[task]
    start = time.perf_counter()
    previous = initial["classic"]
    primary = dict(kind="classic", task=task, params=copy.deepcopy(previous["params"]),
                   tuning=previous["tuning"].copy(), temperature=previous["temperature"],
                   threshold=previous["threshold"])
    with measure_stage("Clássico: refit final em todo outer train"):
        primary["pipeline"] = make_pipeline(task, primary["params"], protocol).fit(
            outer_train[features], outer_train[target])
    primary["fit_seconds"] = time.perf_counter()-start
    experts = {"classic": primary}
    preprocessing = primary["pipeline"].named_steps["preprocessor"]
    X, y = preprocessing.transform(outer_train[features]), outer_train[target].to_numpy()
    for kind in ("Heisenberg",):
        start = time.perf_counter()
        old = initial[kind]
        expert = dict(kind=kind, task=task, preprocessor=preprocessing,
                      config=copy.deepcopy(old["config"]), temperature=old["temperature"],
                      threshold=old["threshold"])
        feature_map = HeisenbergFeatures(expert["config"], factory=factory).fit(X, stage="outer_refit")
        quantum = feature_map.transform(X, stage="outer_refit")
        with measure_stage("Heisenberg: refit ML em todo outer train"):
            model = quantum_estimator(task, old["params"], expert["config"]).fit(
                np.column_stack([quantum, X]), y)
        expert.update(model=HeisenbergExpert(task, feature_map, model),
                      params=copy.deepcopy(old["params"]), tuning=old["tuning"].copy(),
                      inner_calls=copy.deepcopy(old["inner_calls"]))
        expert["fit_seconds"] = time.perf_counter()-start
        experts[kind] = expert
    return experts


@timed_function('Reservas temporais, routers e refit')
def fit_moe_bundle(outer_train, protocol, task, grid, *, heisenberg_config=None,
                   factory=make_heisenberg_map):
    """Ajusta apenas classificação; não recebe qualquer dado do teste externo."""
    if task != "classificacao":
        raise ValueError("Este notebook executa somente classificação.")
    start = time.perf_counter()
    inner, validation, analysis = split_outer_train(outer_train)
    features = protocol["features_numericas"] + protocol["features_categoricas"]
    primary = fit_classic(inner, inner, validation, None, protocol, task, grid)
    primary["kind"] = "classic"
    experts = {"classic": primary}
    for kind, config in (("Heisenberg", heisenberg_config),):
        experts[kind] = fit_secondary(kind, primary, inner, inner, validation, None,
                                      protocol, task, grid, config, factory)
    reserves = {}
    for role, frame in (("validation", validation), ("analysis", analysis)):
        saved = frame[["cnes", "competencia", "mes_alvo"]].copy()
        saved["real"] = frame[TARGETS[task]].to_numpy()
        saved["train_end"] = inner.mes_alvo.max()
        saved["role"] = role
        for name, expert in experts.items():
            raw = expert_output(expert, frame, protocol, stage=role, uncalibrated=True)[0]
            if role == "validation":
                with measure_stage(f"Calibração na validação [{name}]"):
                    expert.update(validation_calibration(raw, saved.real))
            score, prediction = calibrated_prediction(raw, expert)
            saved[f"raw_{name}"] = raw
            saved[f"score_{name}"] = score
            saved[f"prediction_{name}"] = prediction
        reserves[role] = saved.reset_index(drop=True)
    saved_analysis = reserves["analysis"]
    # O transformador dos candidatos não vê nem as features do sexto mês.
    selection_mask = (analysis.mes_alvo < analysis.mes_alvo.max()).to_numpy()
    with measure_stage("Router: pré-processamento de Analysis"):
        selection_preprocessor = make_preprocessor(protocol).fit(analysis.loc[selection_mask, features])
        selection_X = selection_preprocessor.transform(analysis[features])
        router_preprocessor = make_preprocessor(protocol).fit(analysis[features])
        X = router_preprocessor.transform(analysis[features])
    models = {}
    for kind in ("Heisenberg",):
        z = router_targets(task, saved_analysis.real.to_numpy(),
                           saved_analysis.prediction_classic.to_numpy(),
                           saved_analysis[f"prediction_{kind}"].to_numpy())
        saved_analysis[f"z_{kind}"] = z
        started = time.perf_counter()
        gate, diagnostic = fit_router(X, z, analysis.mes_alvo, selection_X=selection_X, selection_z=z)
        diagnostic.update(calibration_end=str(validation.mes_alvo.max()),
                          calibration_n=len(validation), preprocessing_selection_n=int(selection_mask.sum()),
                          preprocessing_refit_n=len(analysis))
        models[f"moe_{kind}"] = dict(router=gate, router_targets=z, router_diagnostics=diagnostic,
            router_fit_seconds=time.perf_counter()-started, router_positive_count=int(z.sum()),
            router_training_size=len(z))
    initial_diagnostics = initial_expert_diagnostics(experts, inner)
    final = refit_experts(outer_train, protocol, task, experts, factory=factory)
    for kind in ("Heisenberg",):
        models[f"moe_{kind}"]["secondary"] = final[kind]
    return dict(classic=final["classic"], models=models, task=task,
                router_preprocessor=router_preprocessor, trained_through=outer_train.mes_alvo.max(),
                validation_predictions=reserves["validation"], analysis_predictions=saved_analysis,
                initial_diagnostics=initial_diagnostics, fit_seconds=time.perf_counter()-start,
                roles={name: dict(n_rows=len(frame), n_months=frame.mes_alvo.nunique(),
                                 start=str(frame.mes_alvo.min().date()), end=str(frame.mes_alvo.max().date()))
                       for name, frame in (("inner_train", inner), ("inner_validation", validation),
                                           ("analysis", analysis), ("final_refit", outer_train))})


def validate_internal_predictions(result, data, folds):
    """Audita reservas, alvos, probabilidades e z antes da exportação."""
    validate_fold_reservations(data, folds)
    used = set(result["metrics"].fold)
    names = ("classic", "Heisenberg")
    for role, key, block in (("validation", "validation_predictions", "inner_validation"),
                             ("analysis", "analysis_predictions", "analysis")):
        saved = result.get(key)
        required = {"fold", "task", "role", "cnes", "competencia", "mes_alvo", "train_end", "real"}
        required.update(f"{prefix}_{name}" for name in names for prefix in ("raw", "score", "prediction"))
        if saved is None or not required.issubset(saved.columns):
            raise ValueError("Previsões internas ausentes ou incompletas.")
        if (saved.duplicated(["fold", "cnes", "mes_alvo"]).any() or set(saved.fold) != used
                or set(saved.task) != {"classificacao"} or set(saved.role) != {role}):
            raise ValueError("Previsões internas duplicadas ou de outro protocolo.")
        for fold in folds:
            if fold["fold"] not in used:
                continue
            actual = saved.loc[saved.fold.eq(fold["fold"])]
            expected = data.iloc[fold[block]]
            joined = expected.merge(actual, on=["cnes", "mes_alvo"], validate="one_to_one", suffixes=("_data", "_saved"))
            if (len(joined) != len(expected) or len(actual) != len(expected)
                    or not np.array_equal(joined.pressao_prox_mes, joined.real)
                    or not np.array_equal(joined.competencia_data, joined.competencia_saved)
                    or not pd.to_datetime(actual.train_end).eq(data.iloc[fold["inner_train"]].mes_alvo.max()).all()
                    or not (pd.to_datetime(actual.train_end) < pd.to_datetime(actual.mes_alvo)).all()):
                raise ValueError("Previsões internas incompatíveis com datas, amostras ou alvos.")
        values = saved[[f"{prefix}_{name}" for name in names for prefix in ("raw", "score")]].to_numpy()
        if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():
            raise ValueError("Probabilidades internas inválidas.")
        for name in names:
            if not np.isin(saved[f"prediction_{name}"], [0, 1]).all():
                raise ValueError("Classes internas inválidas.")
        if role == "analysis":
            for name in ("Heisenberg",):
                expected_z = router_targets("classificacao", saved.real.to_numpy(),
                    saved.prediction_classic.to_numpy(), saved[f"prediction_{name}"].to_numpy())
                if f"z_{name}" not in saved or not np.array_equal(saved[f"z_{name}"], expected_z):
                    raise ValueError("Alvos z incompatíveis com as previsões de Analysis.")


## Swept configurations

The reference uses 19 qubits, 54 PCA components, two layers, α=0.1 and hybrid TreeSHAP top 50. Each setting retains its own full 27-candidate classical and hybrid parameter search. PCA dimensions below capacity are padded by the Heisenberg encoder; dimensions are never repeated.

In [14]:
SWEEP = {
    'reference': dict(q_enc=19,n_components=54,R=2,alpha=0.1,hybrid_shap_top_k=50),
    'qubits_10': dict(q_enc=10,n_components=27,R=2,alpha=0.1,hybrid_shap_top_k=50),
    'qubits_26': dict(q_enc=26,n_components=75,R=2,alpha=0.1,hybrid_shap_top_k=50),
    'alpha_1': dict(q_enc=19,n_components=54,R=2,alpha=1.0,hybrid_shap_top_k=50),
    'layers_3': dict(q_enc=19,n_components=54,R=3,alpha=0.1,hybrid_shap_top_k=50),
    'pca_16': dict(q_enc=19,n_components=16,R=2,alpha=0.1,hybrid_shap_top_k=50),
    'pca_32': dict(q_enc=19,n_components=32,R=2,alpha=0.1,hybrid_shap_top_k=50),
    'shap_20': dict(q_enc=19,n_components=54,R=2,alpha=0.1,hybrid_shap_top_k=20),
    'shap_80': dict(q_enc=19,n_components=54,R=2,alpha=0.1,hybrid_shap_top_k=80),
}
for setting in SWEEP.values():
    assert 0 < setting['n_components'] <= 3 * (setting['q_enc'] - 1)
assert len(SWEEP) * len(folds) == 90
pd.DataFrame(SWEEP).T

,q_enc,n_components,R,alpha,hybrid_shap_top_k
reference,19.0,54.0,2.0,0.1,50.0
qubits_10,10.0,27.0,2.0,0.1,50.0
qubits_26,26.0,75.0,2.0,0.1,50.0
alpha_1,19.0,54.0,2.0,1.0,50.0
layers_3,19.0,54.0,3.0,0.1,50.0
pca_16,19.0,16.0,2.0,0.1,50.0
pca_32,19.0,32.0,2.0,0.1,50.0
shap_20,19.0,54.0,2.0,0.1,20.0
shap_80,19.0,54.0,2.0,0.1,80.0


## Fold-local reuse and parallel searches

Each temporal fold starts with an empty in-memory cache. The search retains all 27 configurations, every training row, and distinct models and routers for each candidate. Two concurrent fits use two XGBoost threads each.

In [15]:
# Fold-local reuse. These objects are discarded at each temporal fold boundary.
from joblib import Parallel, delayed
import xgboost as xgb
FOLD_CACHE = {}
GRID_WORKERS = int(os.environ.get('PQFM_GRID_WORKERS', '2'))
THREADS_PER_FIT = int(os.environ.get('PQFM_XGB_THREADS', '2'))
_original_fit_classic = fit_classic
_original_make_estimator = make_estimator

def _representation_key(config):
    cfg = dict(HEISENBERG_CONFIG, **(config or {}))
    cfg.pop('hybrid_shap_top_k', None)
    return json.dumps(cfg, sort_keys=True, default=str)

def _fit_prepared_xgboost(model, training_matrix):
    # The same quantile cuts and labels are reused by independent histogram fits.
    model._Booster = xgb.train(model.get_xgb_params(), training_matrix,
                               num_boost_round=model.get_params()['n_estimators'])
    model.n_classes_ = 2
    return model

def _grid_estimator(task, params):
    return _original_make_estimator(task, params).set_params(n_jobs=THREADS_PER_FIT)

def _parallel_grid(jobs):
    # Limit independent search fits and estimator threads; keep input ordering for ties.
    with threadpool_limits(limits=THREADS_PER_FIT):
        return Parallel(n_jobs=GRID_WORKERS, prefer='threads')(jobs)

def tune_classic(data, fold, protocol, task, grid):
    features = protocol['features_numericas'] + protocol['features_categoricas']
    target = TARGETS[task]
    train, validation = (data.iloc[fold[role]] for role in ('inner_train', 'inner_validation'))
    if any(block[target].nunique() != 2 for block in (train, validation)):
        raise ValueError('Treino e validação de classificação exigem duas classes.')
    preprocessor = make_preprocessor(protocol).fit(train[features])
    Xt, Xv = preprocessor.transform(train[features]), preprocessor.transform(validation[features])
    training_matrix = xgb.QuantileDMatrix(Xt, label=train[target], nthread=THREADS_PER_FIT)
    def evaluate(params):
        started = time.perf_counter()
        model = _fit_prepared_xgboost(_grid_estimator(task, params), training_matrix)
        score = average_precision_score(validation[target], model.predict_proba(Xv)[:, 1])
        if not np.isfinite(score): raise ValueError('Métrica interna não finita.')
        return dict(params=params, score=float(score), seconds=time.perf_counter()-started)
    rows = _parallel_grid(delayed(evaluate)(params) for params in ParameterGrid(grid))
    if not rows: raise ValueError('Grid vazio.')
    return dict(max(rows, key=lambda row: row['score'])['params']), pd.DataFrame(rows)

def fit_classic(expert, inner_train, inner_validation, calibration, protocol, task, grid):
    if calibration is not None:
        raise ValueError('Cache requer calibração separada na validação interna.')
    if 'initial_classic' not in FOLD_CACHE:
        FOLD_CACHE['initial_classic'] = _original_fit_classic(expert, inner_train, inner_validation, calibration, protocol, task, grid)
    else:
        print('Reusing classical training and 27-fit search for this fold.', flush=True)
    return dict(FOLD_CACHE['initial_classic'])

def tune_heisenberg(inner_train, inner_validation, protocol, task, grid, config=None, factory=make_heisenberg_map):
    features = protocol['features_numericas'] + protocol['features_categoricas']
    target = TARGETS[task]
    cfg = dict(HEISENBERG_CONFIG, **(config or {}))
    key = _representation_key(cfg)
    cached = FOLD_CACHE.get('inner_quantum')
    if cached is None or cached['key'] != key:
        preprocessing = FOLD_CACHE['initial_classic']['pipeline'].named_steps['preprocessor']
        Xt, Xv = preprocessing.transform(inner_train[features]), preprocessing.transform(inner_validation[features])
        validate_task_data(Xt, inner_train[target], task)
        if inner_validation[target].nunique() != 2: raise ValueError('Validação exige ambas as classes.')
        fmap = HeisenbergFeatures(cfg, factory=factory).fit(Xt, stage='inner_train')
        qt = np.column_stack([fmap.transform(Xt, stage='inner_train'), Xt])
        qv = np.column_stack([fmap.transform(Xv, stage='inner_validation'), Xv])
        scaler = StandardScaler().fit(qt)
        st, sv = scaler.transform(qt), scaler.transform(qv)
        # All widths in this representation are evaluated together, each on all rows.
        widths = sorted({dict(HEISENBERG_CONFIG, **c)['hybrid_shap_top_k']
                         for c in SWEEP.values() if _representation_key(c) == key})
        y, yv = inner_train[target].to_numpy(), inner_validation[target].to_numpy()
        training_matrix = xgb.QuantileDMatrix(st, label=y, nthread=THREADS_PER_FIT)
        def evaluate(params):
            started = time.perf_counter()
            estimator = _grid_estimator(task, params)
            selector = HybridSHAPSelector(estimator, top_k=max(widths)).fit(st, y, training_matrix=training_matrix)
            results = {}
            for width in widths:
                idx = selector.ranking_[:min(width, st.shape[1])]
                model = clone(estimator).fit(st[:, idx], y)
                score = average_precision_score(yv, model.predict_proba(sv[:, idx])[:, 1])
                if not np.isfinite(score): raise ValueError('Métrica interna não finita.')
                results[width] = dict(params=params, score=float(score), seconds=time.perf_counter()-started)
            return results
        with measure_stage('Heisenberg: grid paralelo e ranking SHAP compartilhado'):
            records = _parallel_grid(delayed(evaluate)(params) for params in ParameterGrid(grid))
        if not records: raise ValueError('Grid quântico vazio.')
        tables = {width: pd.DataFrame([record[width] for record in records]) for width in widths}
        cached = dict(key=key, fmap=fmap, qt=qt, tables=tables, calls=copy.deepcopy(fmap.calls))
        FOLD_CACHE['inner_quantum'] = cached
    else:
        print('Reusing inner quantum features and TreeSHAP rankings for SHAP-width sweep.', flush=True)
    table = cached['tables'][cfg['hybrid_shap_top_k']]
    best = dict(max(table.to_dict('records'), key=lambda row: row['score'])['params'])
    return best, table.copy(), copy.deepcopy(cached['calls'])

def fit_secondary(kind, primary, expert, inner_train, inner_validation, calibration, protocol, task, grid, config=None, factory=make_heisenberg_map):
    if kind != 'Heisenberg' or calibration is not None: raise ValueError('Configuração secundária inesperada.')
    started = time.perf_counter()
    params, tuning, inner_calls = tune_heisenberg(inner_train, inner_validation, protocol, task, grid, config, factory)
    cached = FOLD_CACHE['inner_quantum']
    fmap = cached['fmap']
    with measure_stage('Heisenberg: reajuste final do estimador ML'):
        model = quantum_estimator(task, params, dict(HEISENBERG_CONFIG, **(config or {}))).fit(cached['qt'], expert[TARGETS[task]].to_numpy())
    return dict(kind=kind, task=task, preprocessor=primary['pipeline'].named_steps['preprocessor'],
                model=HeisenbergExpert(task, fmap, model), params=params, tuning=tuning,
                inner_calls=inner_calls, config=dict(fmap.config, hybrid_shap_top_k=dict(HEISENBERG_CONFIG, **(config or {}))['hybrid_shap_top_k']),
                fit_seconds=time.perf_counter()-started)



def _cached_expert_output(expert, frame, protocol, *, stage, uncalibrated=False):
    # Cache only inference inputs/outputs; no test labels enter training or selection.
    roles = FOLD_CACHE.setdefault('inference_roles', {})
    role = roles.setdefault(stage, {})
    features = protocol['features_numericas'] + protocol['features_categoricas']
    if expert['kind'] == 'classic':
        model = expert['pipeline']
        cached = role.get('classic')
        if cached is None or cached[0] is not model:
            cached = (model, model.predict_proba(frame[features])[:, 1])
            role['classic'] = cached
        raw = cached[1]
    else:
        preprocessing = expert['preprocessor']
        cached = role.get('transformed')
        if cached is None or cached[0] is not preprocessing:
            cached = (preprocessing, preprocessing.transform(frame[features]))
            role['transformed'] = cached
        values = cached[1]
        fmap = expert['model'].feature_map
        quantum = role.get('quantum')
        if quantum is None or quantum[0] is not fmap:
            quantum = (fmap, fmap.transform(values, stage=stage))
            role['quantum'] = quantum
        raw = expert['model'].model.predict_proba(np.column_stack([quantum[1], values]))[:, 1]
    raw = np.asarray(raw, dtype=float)
    if raw.shape != (len(frame),) or not np.isfinite(raw).all() or ((raw < 0) | (raw > 1)).any():
        raise ValueError('Probabilidades inválidas do especialista.')
    if uncalibrated:
        return raw, (raw >= .5).astype(int)
    score = apply_temperature(raw, expert.get('temperature', 1.0))
    return score, (score >= expert.get('threshold', .5)).astype(int)

@timed_function('Experts: refit final sem nova busca')
def refit_experts(outer_train, protocol, task, initial, *, factory=make_heisenberg_map):
    """Reajusta modelos/PCA no outer train; transfere parâmetros, temperatura e threshold."""
    features, target = protocol["features_numericas"] + protocol["features_categoricas"], TARGETS[task]
    start = time.perf_counter()
    previous = initial["classic"]
    primary = dict(kind="classic", task=task, params=copy.deepcopy(previous["params"]),
                   tuning=previous["tuning"].copy(), temperature=previous["temperature"],
                   threshold=previous["threshold"])
    if 'outer_classic_pipeline' not in FOLD_CACHE:
        with measure_stage("Clássico: refit final em todo outer train"):
            FOLD_CACHE['outer_classic_pipeline'] = make_pipeline(task, primary["params"], protocol).fit(
                outer_train[features], outer_train[target])
    primary['pipeline'] = FOLD_CACHE['outer_classic_pipeline']
    primary["fit_seconds"] = time.perf_counter()-start
    experts = {"classic": primary}
    preprocessing = primary["pipeline"].named_steps["preprocessor"]
    if 'outer_classic_matrix' not in FOLD_CACHE:
        FOLD_CACHE['outer_classic_matrix'] = preprocessing.transform(outer_train[features])
    X, y = FOLD_CACHE['outer_classic_matrix'], outer_train[target].to_numpy()
    for kind in ("Heisenberg",):
        start = time.perf_counter()
        old = initial[kind]
        expert = dict(kind=kind, task=task, preprocessor=preprocessing,
                      config=copy.deepcopy(old["config"]), temperature=old["temperature"],
                      threshold=old["threshold"])
        key = _representation_key(expert['config'])
        cached = FOLD_CACHE.get('outer_quantum')
        if cached is None or cached['key'] != key:
            feature_map = HeisenbergFeatures(expert['config'], factory=factory).fit(X, stage='outer_refit')
            quantum = feature_map.transform(X, stage='outer_refit')
            cached = dict(key=key, feature_map=feature_map, quantum=quantum)
            FOLD_CACHE['outer_quantum'] = cached
        feature_map, quantum = cached['feature_map'], cached['quantum']
        with measure_stage("Heisenberg: refit ML em todo outer train"):
            model = quantum_estimator(task, old["params"], expert["config"]).fit(
                np.column_stack([quantum, X]), y)
        expert.update(model=HeisenbergExpert(task, feature_map, model),
                      params=copy.deepcopy(old["params"]), tuning=old["tuning"].copy(),
                      inner_calls=copy.deepcopy(old["inner_calls"]))
        expert["fit_seconds"] = time.perf_counter()-start
        experts[kind] = expert
    return experts


@timed_function('Reservas temporais, routers e refit')
def fit_moe_bundle(outer_train, protocol, task, grid, *, heisenberg_config=None,
                   factory=make_heisenberg_map):
    """Ajusta apenas classificação; não recebe qualquer dado do teste externo."""
    if task != "classificacao":
        raise ValueError("Este notebook executa somente classificação.")
    start = time.perf_counter()
    inner, validation, analysis = split_outer_train(outer_train)
    features = protocol["features_numericas"] + protocol["features_categoricas"]
    primary = fit_classic(inner, inner, validation, None, protocol, task, grid)
    primary["kind"] = "classic"
    experts = {"classic": primary}
    for kind, config in (("Heisenberg", heisenberg_config),):
        experts[kind] = fit_secondary(kind, primary, inner, inner, validation, None,
                                      protocol, task, grid, config, factory)
    reserves = {}
    for role, frame in (("validation", validation), ("analysis", analysis)):
        saved = frame[["cnes", "competencia", "mes_alvo"]].copy()
        saved["real"] = frame[TARGETS[task]].to_numpy()
        saved["train_end"] = inner.mes_alvo.max()
        saved["role"] = role
        for name, expert in experts.items():
            raw = _cached_expert_output(expert, frame, protocol, stage=role, uncalibrated=True)[0]
            if role == "validation":
                with measure_stage(f"Calibração na validação [{name}]"):
                    expert.update(validation_calibration(raw, saved.real))
            score, prediction = calibrated_prediction(raw, expert)
            saved[f"raw_{name}"] = raw
            saved[f"score_{name}"] = score
            saved[f"prediction_{name}"] = prediction
        reserves[role] = saved.reset_index(drop=True)
    saved_analysis = reserves["analysis"]
    # O transformador dos candidatos não vê nem as features do sexto mês.
    selection_mask = (analysis.mes_alvo < analysis.mes_alvo.max()).to_numpy()
    if 'router_features' not in FOLD_CACHE:
        with measure_stage("Router: pré-processamento de Analysis"):
            selection_preprocessor = make_preprocessor(protocol).fit(analysis.loc[selection_mask, features])
            selection_X = selection_preprocessor.transform(analysis[features])
            router_preprocessor = make_preprocessor(protocol).fit(analysis[features])
            X = router_preprocessor.transform(analysis[features])
            FOLD_CACHE['router_features'] = (selection_X, router_preprocessor, X)
    selection_X, router_preprocessor, X = FOLD_CACHE['router_features']
    models = {}
    for kind in ("Heisenberg",):
        z = router_targets(task, saved_analysis.real.to_numpy(),
                           saved_analysis.prediction_classic.to_numpy(),
                           saved_analysis[f"prediction_{kind}"].to_numpy())
        saved_analysis[f"z_{kind}"] = z
        started = time.perf_counter()
        gate, diagnostic = fit_router(X, z, analysis.mes_alvo, selection_X=selection_X, selection_z=z)
        diagnostic.update(calibration_end=str(validation.mes_alvo.max()),
                          calibration_n=len(validation), preprocessing_selection_n=int(selection_mask.sum()),
                          preprocessing_refit_n=len(analysis))
        models[f"moe_{kind}"] = dict(router=gate, router_targets=z, router_diagnostics=diagnostic,
            router_fit_seconds=time.perf_counter()-started, router_positive_count=int(z.sum()),
            router_training_size=len(z))
    initial_diagnostics = initial_expert_diagnostics(experts, inner)
    final = refit_experts(outer_train, protocol, task, experts, factory=factory)
    for kind in ("Heisenberg",):
        models[f"moe_{kind}"]["secondary"] = final[kind]
    return dict(classic=final["classic"], models=models, task=task,
                router_preprocessor=router_preprocessor, trained_through=outer_train.mes_alvo.max(),
                validation_predictions=reserves["validation"], analysis_predictions=saved_analysis,
                initial_diagnostics=initial_diagnostics, fit_seconds=time.perf_counter()-start,
                roles={name: dict(n_rows=len(frame), n_months=frame.mes_alvo.nunique(),
                                 start=str(frame.mes_alvo.min().date()), end=str(frame.mes_alvo.max().date()))
                       for name, frame in (("inner_train", inner), ("inner_validation", validation),
                                           ("analysis", analysis), ("final_refit", outer_train))})



## First-fold execution benchmark

Set `PQFM_SIH_FIRST_FOLD_HILL=1` before starting the remote kernel or loading this notebook. Use `PQFM_SIH_HILL_RESULTS` to choose a JSONL journal. To replay the full-grid finalist, set `PQFM_SIH_HILL_PROFILES=split_analysis_full,split_analysis_full_repeat`.

The benchmark uses March 2025 only. Each profile records the validation grid, held-out AP, stage spans, and Linux and GPU counters. Staged profiles promote candidates using inner-validation AP. The full grid evaluates all 27 settings through 300 rounds. The fold timer starts after data loading, which is recorded separately.

Set `PQFM_SIH_AOT` to the exported PCA-54 artifact directory to run the precompiled-only comparison. The four profiles toggle block covariance, separate outer training and test extraction with GPU matrix assembly, and router pre-binning in sequence. Use a new journal for this comparison.


In [ ]:
# Opt-in execution benchmark.
import math
import resource
import subprocess
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager, nullcontext
from queue import Empty, Queue
from threading import Event, Lock, Thread


class FirstFoldRun:
    def __init__(self, settings):
        import cupy
        import torch
        import xgboost
        from pqfmlib.experimental.heisenberg_tilelang import TileLangExecutor

        self.cp, self.torch, self.xgb = cupy, torch, xgboost
        self.executor_type = TileLangExecutor
        self.settings = settings
        self.artifact = (Path(os.environ['PQFM_SIH_AOT']) / settings.get('artifact', '')
                         if settings.get('aot') else None)
        self.aot_runners = {}
        if self.artifact and not all((self.artifact / name).is_file()
                                     for name in ('metadata.json', 'coefficients.so', 'evolve_and_measure.so')):
            raise FileNotFoundError(f'Exported TileLang artifacts are required at {self.artifact}')
        self.grid = list(ParameterGrid(GRID_CLASSIFICATION))
        if len(self.grid) != 27 or torch.cuda.device_count() < 6:
            raise ValueError('The benchmark requires 27 settings and six visible GPUs.')
        self.lock = Lock()
        self.events = []
        self.selected = {}
        self.counts = Counter()
        self.threads = settings.get('threads', 4)
        self.config = dict(HEISENBERG_CONFIG, **SWEEP['shap_80']) | settings.get('encoding_config', {})
        self.features = protocol['features_numericas'] + protocol['features_categoricas']

    @contextmanager
    def stage(self, name, device=None, **metadata):
        begin = time.perf_counter()
        cpu = time.thread_time()
        try:
            yield
        finally:
            with self.lock:
                self.events.append(dict(name=name, gpu=device, start=begin-self.start,
                    stop=time.perf_counter()-self.start, calling_thread_cpu=time.thread_time()-cpu, **metadata))

    def device(self, device):
        return nullcontext() if device is None else self.cp.cuda.Device(device)

    def train(self, params, X, y, device, rounds=300, matrix=None, threads=None):
        estimator = make_estimator('classificacao', params).set_params(
            n_estimators=rounds, n_jobs=self.threads if threads is None else threads,
            device='cpu' if device is None else f'cuda:{device}')
        return estimator.fit(X, y) if matrix is None else _fit_prepared_xgboost(estimator, matrix)

    def quantile(self, X, y):
        return self.xgb.QuantileDMatrix(X, label=y, nthread=self.threads)

    def gpu_array(self, X, device):
        return X if device is None else self.cp.asarray(X, dtype=self.cp.float32)

    def host(self, X):
        return self.cp.asnumpy(X) if isinstance(X, self.cp.ndarray) else X

    def shap_scores(self, booster, X, device):
        def reduce_rows(model, rows, gpu):
            model.set_param({'device': 'cpu' if gpu is None else f'cuda:{gpu}'})
            total = np.zeros(rows.shape[1], dtype=np.float64)
            for start in range(0, len(rows), 8192):
                batch = self.gpu_array(rows[start:start+8192], gpu)
                # Booster.predict returns host values. inplace_predict has no TreeSHAP mode.
                matrix = self.xgb.DMatrix(batch, nthread=self.threads)
                values = model.predict(matrix, pred_contribs=True)[:, :-1]
                total += np.abs(values).sum(axis=0, dtype=np.float64)
            return total

        if not isinstance(device, tuple):
            with self.device(device):
                return reduce_rows(booster, X, device) / len(X)

        # Each GPU gets an independent booster and nonoverlapping training rows.
        # Serialize once in memory. XGBoost exposes contributions on the host.
        payload = booster.save_raw()
        boundaries = np.linspace(0, len(X), len(device) + 1, dtype=int)

        def shard(item):
            idx, gpu = item
            with self.device(gpu), self.stage('exact_shap_shard', gpu):
                model = self.xgb.Booster(model_file=payload)
                return reduce_rows(model, X[boundaries[idx]:boundaries[idx+1]], gpu)

        with ThreadPoolExecutor(max_workers=len(device)) as pool:
            return np.sum(list(pool.map(shard, enumerate(device))), axis=0) / len(X)

    def mask(self, model, X, device):
        scores = self.shap_scores(model.get_booster(), X, device)
        return np.argsort(-scores, kind='stable')[:self.config['hybrid_shap_top_k']]

    def search(self, X, V, y, vy, hybrid, gpu_ids):
        stage = 'hybrid' if hybrid else 'classic'
        # Cache quantization by device and ordered column mask. Shared matrices
        # are immutable after construction, and each fit owns its Booster.
        matrices = {}
        reduced = {}
        arrays = {}
        matrix_lock = Lock()
        branch = 'hybrid' if hybrid else 'classic'
        cpu_workers = self.settings.get(branch + '_workers', self.settings['workers'])
        layout = [None] * cpu_workers + list(gpu_ids) * self.settings.get('lanes', 1)

        preparation_locks = ({device: Lock() for device in layout}
                             if self.settings.get('parallel_search_bins') else {})

        def prepared(device):
            with preparation_locks.get(device, matrix_lock):
                if device not in arrays:
                    arrays[device] = (self.gpu_array(X, device), self.gpu_array(V, device))
                if device not in matrices:
                    with self.stage(stage + '_search_prebin', device):
                        matrices[device] = self.quantile(arrays[device][0], y)
                return arrays[device], matrices[device]

        def selected_matrix(device, Xt, take):
            key = (device, tuple(take))
            with matrix_lock:
                if key in reduced:
                    with self.lock:
                        self.counts['mask_matrix_hits'] += 1
                    return reduced[key]
            gathered = Xt[:, take]
            result = self.quantile(gathered, y)
            with matrix_lock:
                # Limit retained histograms. Fits keep their own local reference.
                if len(reduced) >= 2 * len(set(layout)):
                    reduced.pop(next(iter(reduced)))
                return reduced.setdefault(key, result)

        def evaluate(index, rounds, device):
            with self.device(device), self.stage(stage + '_candidate', device, candidate=index, rounds=rounds):
                (Xt, Vt), matrix = prepared(device)
                params = self.grid[index]
                with self.stage(stage + '_candidate_fit', device, candidate=index, rounds=rounds):
                    model = self.train(params, Xt, y, device, rounds, matrix)
                if hybrid:
                    with self.stage('hybrid_candidate_shap', device, candidate=index, rounds=rounds):
                        take = self.mask(model, Xt, device)
                    with self.stage('hybrid_candidate_selected_bins', device, candidate=index, rounds=rounds):
                        masked_matrix = selected_matrix(device, Xt, take)
                    with self.stage('hybrid_candidate_selected_fit', device, candidate=index, rounds=rounds):
                        model = self.train(params, None, y, device, rounds, masked_matrix)
                    Vt = Vt[:, take]
                prediction = self.host(model.predict_proba(Vt)[:, 1])
                return index, float(average_precision_score(vy, prediction)), device

        # Keep candidate placement independent of thread races. Workers pull
        # longest-first tasks from a shared queue for their assigned device family.
        n_gpu = len(layout) - cpu_workers
        gpu_budget = len(self.grid) if not cpu_workers else round(
            len(self.grid) * 3 * n_gpu / (cpu_workers + 3 * n_gpu))
        placement = {index for index in range(len(self.grid))
                     if (index * 17) % len(self.grid) < gpu_budget}

        def batch(indexes, rounds, pool):
            queues = {False: Queue(), True: Queue()}
            for index in sorted(indexes, key=lambda k: -self.grid[k]['max_depth']):
                queues[index in placement].put(index)

            def lane(device, queue, first):
                results = [evaluate(first, rounds, device)]
                while True:
                    try:
                        index = queue.get_nowait()
                    except Empty:
                        return results
                    results.append(evaluate(index, rounds, device))

            jobs = []
            for device in layout:
                queue = queues[device is not None]
                try:
                    index = queue.get_nowait()
                except Empty:
                    continue
                jobs.append(pool.submit(lane, device, queue, index))
            return [row for job in jobs for row in job.result()]

        with self.stage(stage + '_search'), ThreadPoolExecutor(max_workers=len(layout)) as pool:
            checkpoint = self.settings.get('checkpoint')
            if checkpoint:
                preliminary = batch(range(len(self.grid)), checkpoint, pool)
                promoted = sorted(preliminary, key=lambda row: (-row[1], row[0]))[:self.settings['promote']]
                indexes = [index for index, _, _ in promoted]
            else:
                preliminary = []
                indexes = self.settings.get('search_candidates', {}).get(stage, range(len(self.grid)))
            results = batch(indexes, 300, pool)
        winner, score, _ = min(results, key=lambda row: (-row[1], row[0]))
        self.selected[stage] = dict(params=self.grid[winner], validation_ap=score,
                                    round300=sorted(results), checkpoint=sorted(preliminary))
        return self.grid[winner]

    def quantum(self, fmap, X, name):
        with self.stage(name):
            if self.artifact is None:
                return fmap.transform(X, stage=name)
            from pqfmlib.experimental.heisenberg_sih_aot import SIHHeisenbergAOT

            with self.stage(name + '_projection'):
                pca = fmap.pca_.transform(X)
            devices = self.quantum_context.devices
            parts = np.array_split(pca, min(len(devices), len(pca)))

            def run(item):
                device, part = item
                with self.torch.cuda.device(device):
                    key = (fmap, device)
                    if key not in self.aot_runners:
                        with self.stage('aot_load', device):
                            self.aot_runners[key] = SIHHeisenbergAOT(fmap.map_, self.artifact)
                    with self.stage('aot_inference', device):
                        return self.aot_runners[key].transform(part)

            with ThreadPoolExecutor(max_workers=len(parts)) as pool:
                return np.concatenate(list(pool.map(run, zip(devices, parts))))

    @contextmanager
    def sharded_quantum(self):
        cls = self.executor_type
        original = cls.execute
        active = self

        def execute(executor, prepared, theta):
            if active.artifact is not None:
                raise RuntimeError('AOT benchmark forbids TileLang JIT execution.')
            devices = getattr(active.quantum_context, 'devices', (0, 1, 2))
            chunk = 320
            count = min(len(devices), math.ceil(len(theta) / chunk))
            tiles = math.ceil(len(theta) / chunk)
            ends = [chunk * (tiles * i // count) for i in range(count)] + [len(theta)]

            def run(i):
                device = devices[i]
                part = theta[ends[i]:ends[i+1]]
                size = len(part)
                extra = -size % chunk
                if extra:
                    part = np.concatenate([part, np.repeat(part[-1:], extra, axis=0)])
                with active.torch.cuda.device(device), active.stage('quantum_gpu', device):
                    values, metadata = original(cls(), prepared, part)
                return values[:size], metadata

            with ThreadPoolExecutor(max_workers=count) as pool:
                shards = list(pool.map(run, range(count)))
            return np.concatenate([values for values, _ in shards]), shards[0][1]

        from threading import local
        self.quantum_context = local()
        cls.execute = execute
        try:
            yield
        finally:
            cls.execute = original

    def representation(self, raw, phase, devices, after_fit):
        self.quantum_context.devices = devices
        with self.stage(phase + '_pca'):
            fmap = HeisenbergFeatures(self.config).fit(
                raw[phase], stage=phase, pca_threads=self.settings.get('pca_threads', 4),
                structure_only=self.settings.get('structure_only', False),
                categorical_start=len(protocol['features_numericas']) if self.settings.get('block_pca') else None,
                stable_pca_sign=self.settings.get('stable_pca_sign', False))
        after_fit()  # GPU shards become available after the inner representation.
        encoded = {'feature_map': fmap}
        for name, matrix in raw.items():
            if name == 'outer_test' and self.settings.get('split_outer'):
                continue
            values = self.quantum(fmap, matrix, name)
            with self.stage(name + '_gather'):
                encoded[name] = ((values, matrix) if self.settings.get('split_outer')
                                 else np.column_stack([values, matrix]))
        return encoded

    def refit(self, params, X, y, device, phase):
        with self.stage(phase + '_quantile', device):
            matrix = self.xgb.QuantileDMatrix(X, label=y, nthread=8)
        with self.stage(phase + '_boosting', device):
            return self.train(params, None, y, device, matrix=matrix, threads=8)

    def hybrid_fit(self, params, X, y, phase, classical_stats=None):
        rank_device = self.settings['rank'] if phase == 'outer' else self.settings.get('inner', None)
        fit_device = self.settings['fit'] if phase == 'outer' else self.settings.get('inner', None)
        resident = self.settings.get('resident', False) and rank_device == fit_device == 3
        with self.stage(phase + '_scale', rank_device if resident else None):
            if isinstance(X, tuple):
                with self.device(rank_device):
                    with self.stage('outer_scaler_statistics'):
                        stats = [StandardScaler().fit(X[0]), classical_stats]
                    scaler = StandardScaler()
                    for attribute in ('mean_', 'var_', 'scale_'):
                        setattr(scaler, attribute, np.concatenate([getattr(model, attribute) for model in stats]))
                    scaler.n_features_in_ = sum(block.shape[1] for block in X)
                    scaler.n_samples_seen_ = len(X[0])
                    rank_X = self.cp.empty((len(X[0]), scaler.n_features_in_), dtype=self.cp.float32)
                    offset = 0
                    for block, model in zip(X, stats):
                        end = offset + block.shape[1]
                        with self.stage('outer_scaled_gpu_assembly', rank_device):
                            mean, scale = self.cp.asarray(model.mean_), self.cp.asarray(model.scale_)
                            for start in range(0, len(block), 8192):
                                batch = self.cp.asarray(block[start:start+8192])
                                rank_X[start:start+len(batch), offset:end] = (batch - mean) / scale
                            self.cp.cuda.get_current_stream().synchronize()
                        offset = end
                    scaled = None
            elif resident and phase == 'outer' and self.settings.get('gpu_scaler'):
                with self.device(rank_device):
                    array = self.cp.asarray(X, dtype=self.cp.float64)
                    mean = array.mean(axis=0)
                    variance = array.var(axis=0)
                    eps = np.finfo(np.float64).eps
                    constant = variance <= len(X) * eps * variance + (len(X) * mean * eps) ** 2
                    scale = self.cp.sqrt(self.cp.where(constant, 1, variance))
                    scaler = StandardScaler()
                    scaler.mean_, scaler.var_, scaler.scale_ = map(self.host, (mean, variance, scale))
                    scaler.n_features_in_, scaler.n_samples_seen_ = X.shape[1], len(X)
                    array -= mean
                    array /= scale
                    rank_X = array.astype(self.cp.float32)
                    del array
                scaled = None
            else:
                scaler = StandardScaler().fit(X)
                scaled = scaler.transform(X)
                with self.device(rank_device):
                    rank_X = self.gpu_array(scaled, rank_device)
        with self.device(rank_device), self.stage(phase + '_ranking_fit', rank_device):
            ranking = self.refit(params, rank_X, y, rank_device, phase + '_ranking')
        shap_device = (tuple(self.settings.get('shap_devices', (3,))) if phase == 'outer' and not resident
                       else (4,) if phase == 'inner' else (3,))
        shap_device = shap_device[0] if len(shap_device) == 1 else shap_device
        with self.stage(phase + '_exact_shap', shap_device):
            take = self.mask(ranking, rank_X if resident else scaled, shap_device)
        with self.device(fit_device):
            with self.stage(phase + '_feature_gather', fit_device):
                reduced = rank_X[:, take] if resident else self.gpu_array(scaled[:, take], fit_device)
            with self.stage(phase + '_prediction_fit', fit_device):
                model = self.refit(params, reduced, y, fit_device, phase + '_prediction')
        self.selected[phase + '_mask'] = take.tolist()
        return scaler, take, model, fit_device

    def predict(self, fitted, X, phase):
        scaler, take, model, device = fitted
        with self.device(device), self.stage(phase + '_prediction', device):
            selected = scaler.transform(X)[:, take]
            return self.host(model.predict_proba(self.gpu_array(selected, device))[:, 1])

    def run(self, fold):
        outer = data.loc[fold['train']]
        test = data.loc[fold['test']]
        inner, validation, analysis = split_outer_train(outer)
        y, vy = inner.morte.to_numpy(), validation.morte.to_numpy()
        with self.sharded_quantum(), ThreadPoolExecutor(max_workers=6) as pool:
            # CPU classical search starts as soon as the common classical preprocessing
            # is ready. Separate mapping/estimator state stays local to each branch.
            pre = make_preprocessor(protocol)
            with self.stage('inner_classic_preprocessing'):
                X = pre.fit_transform(inner[self.features])
                V = pre.transform(validation[self.features])
            classic_gpus = tuple(self.settings.get('classic_gpus', (3, 4, 5)))
            classic = pool.submit(self.search, X, V, y, vy, False, classic_gpus)

            inner_ready = Event()

            def inner_quantum():
                self.quantum_context.devices = (0, 1, 2)
                with self.stage('inner_pca'):
                    fmap = HeisenbergFeatures(self.config).fit(
                        X, stage='inner_train', structure_only=self.settings.get('structure_only', False))
                inner_ready.set()
                roles = [('inner_train', X), ('validation', V)]
                if not self.settings.get('split_analysis'):
                    roles.append(('analysis', pre.transform(analysis[self.features])))
                return fmap, {name: np.column_stack([self.quantum(fmap, raw, name), raw])
                              for name, raw in roles}

            inner_maps = pool.submit(inner_quantum)
            outer_y = outer.morte.to_numpy()

            def prepare_outer():
                with self.stage('outer_preprocess'):
                    preprocessor = make_preprocessor(protocol)
                    return {'outer_refit': preprocessor.fit_transform(outer[self.features]),
                            'outer_test': preprocessor.transform(test[self.features])}

            def outer_quantum():
                if self.settings.get('early_pca'):
                    while not inner_ready.wait(0.1):
                        if inner_maps.done():
                            inner_maps.result()
                else:
                    inner_maps.result()
                return self.representation(outer_raw.result(), 'outer_refit', (0, 1, 2), inner_maps.result)

            def final_classic():
                params = classic.result()
                arrays = outer_raw.result()
                device = self.settings.get('classic_refit')
                with self.device(device), self.stage('classic_outer_fit_and_prediction', device):
                    matrix = self.gpu_array(arrays['outer_refit'], device)
                    model = self.refit(params, matrix, outer_y, device, 'classic_outer')
                    return self.host(model.predict_proba(self.gpu_array(arrays['outer_test'], device))[:, 1])

            outer_raw = pool.submit(prepare_outer)
            def prepare_outer_stats():
                with self.stage('outer_classical_scaler_statistics'):
                    return StandardScaler().fit(outer_raw.result()['outer_refit'])

            outer_stats = pool.submit(prepare_outer_stats) if self.settings.get('split_outer') else None
            outer_maps = pool.submit(outer_quantum)
            final_c = pool.submit(final_classic)

            def outer_test_quantum():
                fitted = outer_maps.result()
                if not self.settings.get('split_outer'):
                    return fitted['outer_test']
                self.quantum_context.devices = (0, 1, 2)
                matrix = outer_raw.result()['outer_test']
                return np.column_stack([self.quantum(fitted['feature_map'], matrix, 'outer_test'), matrix])

            outer_test_map = pool.submit(outer_test_quantum)
            with self.stage('router_preprocessing'):
                mask = (analysis.mes_alvo < analysis.mes_alvo.max()).to_numpy()
                router_select = make_preprocessor(protocol).fit(analysis.loc[mask, self.features])
                router_final = make_preprocessor(protocol).fit(analysis[self.features])
                selection_X = router_select.transform(analysis[self.features])
                router_X = router_final.transform(analysis[self.features])
            def prepare_router_bins():
                devices = self.settings['router_gpus']
                matrices = {}
                roles = [('selection', selection_X[mask]), ('refit', router_X)]
                with ThreadPoolExecutor(max_workers=len(devices)) as workers:
                    def prepare(device):
                        with self.device(device):
                            result = {}
                            for role, values in roles:
                                with self.stage('router_' + role + '_prebin', device):
                                    # sklearn's CUDA estimator receives NumPy inputs in
                                    # the reference path, so its cuts come from CPU data.
                                    result[(device, role)] = self.quantile(values, None)
                            return result
                    for result in workers.map(prepare, devices):
                        matrices.update(result)
                return matrices

            router_bins = pool.submit(prepare_router_bins) if self.settings.get('prebin_router') else None
            fmap, maps = inner_maps.result()

            def analysis_features():
                final_c.result()  # The analysis branch uses the classical refit's GPU.
                self.quantum_context.devices = (5,)
                raw = pre.transform(analysis[self.features])
                return raw, np.column_stack([self.quantum(fmap, raw, 'analysis'), raw])

            if self.settings.get('split_analysis'):
                analysis_future = pool.submit(analysis_features)
            else:
                analysis_future = None
            classic_params = classic.result()
            classic_inner = pool.submit(self.train, classic_params, X, y, None, threads=8)
            with self.stage('inner_hybrid_scale'):
                scale = StandardScaler().fit(maps['inner_train'])
                hX, hV = [scale.transform(maps[role]) for role in ('inner_train', 'validation')]
            hybrid_gpus = tuple(self.settings.get('hybrid_gpus', (3, 4, 5)))
            if any(gpu < 3 for gpu in hybrid_gpus) and not self.settings.get('overlap_all_gpus'):
                outer_maps.result()
            hparams = self.search(hX, hV, y, vy, True, hybrid_gpus)
            def final_hybrid():
                with self.stage('outer_wait_for_representation'):
                    matrix = outer_maps.result()['outer_refit']
                return self.hybrid_fit(hparams, matrix, outer_y, 'outer',
                                       classical_stats=outer_stats.result() if outer_stats else None)

            if not self.settings.get('early_inner'):
                outer_maps.result()
            final_h = pool.submit(final_hybrid)
            fitted = self.hybrid_fit(hparams, maps['inner_train'], y, 'inner')
            with self.stage('calibration_and_router_targets'):
                c = classic_inner.result()
                ct = validation_calibration(c.predict_proba(V)[:, 1], vy)
                ht = validation_calibration(self.predict(fitted, maps['validation'], 'validation'), vy)
                if analysis_future is not None:
                    analysis_raw, analysis_matrix = analysis_future.result()
                else:
                    analysis_raw, analysis_matrix = pre.transform(analysis[self.features]), maps['analysis']
                apc = calibrated_prediction(c.predict_proba(analysis_raw)[:, 1], ct)[1]
                aph = calibrated_prediction(self.predict(fitted, analysis_matrix, 'analysis'), ht)[1]
                z = router_targets('classificacao', analysis.morte.to_numpy(), apc, aph)
            def router_candidate(rate):
                model = make_router(rate)
                devices = self.settings.get('router_gpus', ())
                if devices:
                    index = ROUTER_LEARNING_RATE_GRID.index(rate)
                    model.set_params(device=f'cuda:{devices[index % len(devices)]}', n_jobs=2)
                return model

            def fit_router_bins(model, labels, role):
                device = int(model.get_params()['device'].split(':')[1])
                matrix = router_bins.result()[(device, role)]
                with self.device(device), self.stage('router_' + role + '_fit', device):
                    matrix.set_label(labels)
                    return _fit_prepared_xgboost(model, matrix)

            with self.stage('router_fit'):
                gate, diagnostic = fit_router(router_X, z, analysis.mes_alvo,
                                              selection_X=selection_X, selection_z=z,
                                              search_workers=self.settings.get('router_workers', 4),
                                              candidate_factory=router_candidate,
                                              prepared_fit=fit_router_bins if router_bins is not None else None)
            with self.stage('outer_prediction'):
                p = calibrated_prediction(final_c.result(), ct)[0]
                h = calibrated_prediction(self.predict(final_h.result(), outer_test_map.result(), 'outer'), ht)[0]
                w = gate.predict_proba(router_final.transform(test[self.features]))[:, 1]
        scores = {name: float(average_precision_score(test.morte, score))
                  for name, score in [('classic', p), ('hybrid', h), ('moe', (1-w)*p+w*h)]}
        return dict(scores=scores, selected=self.selected, counts=dict(self.counts),
                    predictions=dict(classic=p.tolist(), hybrid=h.tolist(), router=w.tolist()) if self.artifact else None,
                    rows=dict(train=len(outer), test=len(test), inner=len(inner), validation=len(validation),
                              analysis=len(analysis)),
                    classical_widths=dict(inner=X.shape[1], outer=outer_raw.result()['outer_refit'].shape[1]),
                    router_rate=diagnostic['selected_learning_rate'])


def linux_first_fold_snapshot():
    """Process counters and container limits sampled without a profiler dependency."""
    cgroup = Path('/sys/fs/cgroup')

    def fields(path, delimiter=None):
        try:
            records = (line.split(delimiter, 1) for line in Path(path).read_text().splitlines())
            return {k.strip(): int(v.split()[0]) for k, v in records if v.split()[0].isdigit()}
        except (OSError, ValueError, IndexError):
            return {}

    def scalar(name):
        try:
            value = (cgroup / name).read_text().strip()
            return int(value) if value.isdigit() else value
        except OSError:
            return None

    faults = resource.getrusage(resource.RUSAGE_SELF)
    return dict(cpu_seconds=time.process_time(), io=fields('/proc/self/io', ':'),
                status_kib=fields('/proc/self/status', ':'), major_faults=faults.ru_majflt,
                minor_faults=faults.ru_minflt, memory_current=scalar('memory.current'),
                memory_max=scalar('memory.max'), swap_current=scalar('memory.swap.current'),
                memory_stat=fields(cgroup / 'memory.stat'), cpu_stat=fields(cgroup / 'cpu.stat'),
                io_stat=(cgroup / 'io.stat').read_text().strip() if (cgroup / 'io.stat').exists() else None)


def first_fold_trial(name, settings, journal, fold):
    run = FirstFoldRun(settings)
    for device in range(6):
        with run.device(device):
            run.cp.get_default_memory_pool().free_all_blocks()
    snapshots = []
    stop = Event()

    def sample():
        while not stop.is_set():
            now = time.perf_counter() - run.start
            state = linux_first_fold_snapshot()
            try:
                rows = subprocess.check_output(
                    ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used',
                     '--format=csv,noheader,nounits'], text=True, timeout=4)
                gpu = [list(map(int, line.split(','))) for line in rows.splitlines()]
            except (OSError, ValueError, subprocess.SubprocessError):
                gpu = []
            snapshots.append(dict(offset=now, gpu=gpu, memory_current=state['memory_current'],
                                  rss_kib=state['status_kib'].get('VmRSS'),
                                  cpu_seconds=state['cpu_seconds']))
            stop.wait(1)

    before = linux_first_fold_snapshot()
    run.start = time.perf_counter()
    monitor = Thread(target=sample, daemon=True)
    monitor.start()
    try:
        with threadpool_limits(limits=2, user_api='blas'):
            result = run.run(fold)
        elapsed = time.perf_counter() - run.start
    finally:
        stop.set()
        monitor.join()
    after = linux_first_fold_snapshot()
    result.update(name=name, settings=settings, seconds=elapsed, events=run.events,
                  fold=int(fold["fold"]), grid_parameters=list(run.grid),
                  encoding_config=run.config,
                  dataset_load=FIRST_FOLD_DATA_LOAD,
                  resources=dict(before=before, after=after, samples=snapshots,
                                 mean_process_cores=(after['cpu_seconds']-before['cpu_seconds'])/elapsed))
    with journal.open('a') as output:
        output.write(json.dumps(result) + '\n')
    print('HILL_RESULT', name, round(elapsed, 3), result['scores'], flush=True)
    return result


WIDE_ENCODINGS = {
    f'q{q}_{name}': dict(q_enc=q, n_components=50, R=1, alpha=alpha,
                        encoding_mode=mode, balanced_encoding=balanced)
    for q in (26, 33)
    for name, mode, balanced, alpha in (
        ('axes', 'multi_axis', False, .1),
        ('balanced', 'multi_axis', True, .1),
        ('layers', 'shared_feature', False, .2),
    )
}


def compile_wide_encodings(root, batch=320):
    """Offline build. Runtime sweep loads these libraries without compiling."""
    from pqfmlib.experimental.heisenberg_sih_aot import build_sih_kernels, plan_sih_circuit
    for name, overrides in WIDE_ENCODINGS.items():
        config = HEISENBERG_CONFIG | SWEEP['shap_80'] | overrides
        fmap = make_heisenberg_map(config, SEED).fit(np.zeros((1, config['n_components'])))
        shape, *_ = plan_sih_circuit(fmap)
        print('WIDE_SHAPE', name, shape, flush=True)
        build_sih_kernels(Path(root) / name, batch=batch, **shape)
        print('WIDE_BUILT', name, flush=True)


def first_fold_hillclimb():
    import traceback

    journal = Path(os.environ.get('PQFM_SIH_HILL_RESULTS', 'first-fold-hill.jsonl'))
    journal.parent.mkdir(parents=True, exist_ok=True)
    base = dict(workers=8, threads=4, lanes=1, rank=None, fit=None, inner=None)
    profiles = {
        'cpu27x2_hybrid': dict(base, workers=27, threads=2, classic_gpus=[]),
        'cpu16x4_hybrid': dict(base, workers=16, threads=4, classic_gpus=[]),
        'cpu9x7_hybrid': dict(base, workers=9, threads=7, classic_gpus=[]),
        'gpu_rank': dict(base, rank=3),
        'gpu_both_branches': dict(base, rank=3, fit=3, classic_refit=5, resident=True,
                                  classic_gpus=[3, 4], hybrid_gpus=[3, 4]),
        'gpu_prediction': dict(base, fit=3),
        'resident_gpu': dict(base, rank=3, fit=3, resident=True),
        'cpu_classic_gpu_hybrid': dict(base, workers=16, threads=4, classic_gpus=[],
                                       hybrid_workers=0, lanes=2, inner=4, rank=3, fit=3,
                                       resident=True, hybrid_gpus=[0, 1, 2, 3, 4, 5]),
    }
    cuda = dict(profiles['gpu_both_branches'], workers=4, lanes=2, threads=2,
                classic_workers=0, hybrid_workers=0, classic_gpus=[0,1,2,3,4,5],
                hybrid_gpus=[0,1,2,3,4], overlap_all_gpus=True, inner=4)
    profiles.update({
        'cuda_parallel_grid': cuda,
        'cuda_with_router': dict(cuda, router_gpus=[0, 1, 2]),
        'cuda_with_router_promote': dict(cuda, router_gpus=[0, 1, 2], checkpoint=75, promote=6),
    })
    early = dict(profiles['cuda_with_router'], early_pca=True, early_inner=True)
    profiles.update({
        'early_full_grid': early,
        'early_promote': dict(early, checkpoint=75, promote=6),
        'early_promote_shap4': dict(early, checkpoint=75, promote=6,
                                    shap_devices=[0, 1, 2, 3], resident=False, router_gpus=[4, 5]),
    })
    faster = dict(early, checkpoint=75, promote=6, structure_only=True,
                  router_gpus=[0, 1, 2, 4, 5], router_workers=5)
    profiles.update({
        'fast_host': dict(faster, pca_threads=12),
        'fast_device': dict(faster, pca_threads=12, gpu_scaler=True),
        'fast_device_full': dict(faster, pca_threads=12, gpu_scaler=True, checkpoint=None, promote=None),
    })
    split = dict(profiles['fast_device'], split_analysis=True)
    profiles.update({
        'split_analysis_promote': split,
        'split_analysis_full': dict(split, checkpoint=None, promote=None),
        'stable_dense_pca_full': dict(split, checkpoint=None, promote=None, stable_pca_sign=True),
        'block_pca_full': dict(split, checkpoint=None, promote=None, stable_pca_sign=True, block_pca=True),
        'split_analysis_short': dict(split, checkpoint=50, promote=4),
    })
    aot = dict(split, checkpoint=None, promote=None, aot=True, stable_pca_sign=True)
    profiles.update({
        'aot_reference': aot,
        'aot_block_pca': dict(aot, block_pca=True),
        'aot_split_outer': dict(aot, block_pca=True, split_outer=True),
        'aot_prebin_router': dict(aot, block_pca=True, split_outer=True, prebin_router=True),
    })
    profiles['aot_parallel_search_bins'] = dict(profiles['aot_prebin_router'], parallel_search_bins=True)
    records = {row['name']: row for row in
               (map(json.loads, journal.read_text().splitlines()) if journal.exists() else ())
               if 'seconds' in row}

    def trial(name, settings, fold=folds[0]):
        if name in records:
            if records[name]['settings'] != json.loads(json.dumps(settings)):
                raise ValueError(f'Settings changed for existing benchmark {name}. Choose a new results path.')
            return
        print('HILL_START', name, settings, flush=True)
        try:
            records[name] = first_fold_trial(name, settings, journal, fold)
        except Exception as exc:
            traceback.print_exc()
            with journal.open('a') as output:
                output.write(json.dumps(dict(name=name, settings=settings,
                                             error=f'{type(exc).__name__}: {exc}')) + '\n')

    if os.environ.get('PQFM_SIH_WIDE_ENCODINGS') == '1':
        if not os.environ.get('PQFM_SIH_AOT'):
            raise ValueError('The wide sweep requires precompiled kernels.')
        names = []
        for fold in folds:
            for encoding, config in WIDE_ENCODINGS.items():
                name = f"fold{fold['fold']:02d}_{encoding}"
                names.append(name)
                trial(name, dict(profiles['aot_prebin_router'], artifact=encoding,
                                 encoding_config=config), fold)
        missing = [name for name in names if name not in records]
        if missing:
            raise RuntimeError(f'Wide encoding runs failed: {missing}')
        return pd.DataFrame([dict(name=name, seconds=records[name]['seconds'],
                                  **records[name]['scores']) for name in names])

    if os.environ.get('PQFM_SIH_HILL_FOLDS') == 'all':
        if not os.environ.get('PQFM_SIH_AOT'):
            raise ValueError('The full benchmark requires precompiled kernels.')
        profile = 'aot_prebin_router'
        names = []
        for fold in folds:
            name = f"fold{fold['fold']:02d}_{profile}"
            names.append(name)
            trial(name, profiles[profile], fold)
        missing = [name for name in names if name not in records]
        if missing:
            raise RuntimeError(f'Folds failed: {missing}')
        return pd.DataFrame([dict(fold=records[name]['fold'], seconds=records[name]['seconds'],
                                  **records[name]['scores']) for name in names])

    shortlist_history = os.environ.get('PQFM_SIH_HILL_SHORTLIST_HISTORY')
    if shortlist_history:
        if not os.environ.get('PQFM_SIH_AOT'):
            raise ValueError('The shortlist benchmark requires precompiled kernels.')
        target_fold = int(os.environ['PQFM_SIH_HILL_FOLD'])
        fold = next(fold for fold in folds if fold['fold'] == target_fold)
        prior = {row['fold']: row for row in map(json.loads, Path(shortlist_history).read_text().splitlines())
                 if 'seconds' in row and row['fold'] < target_fold}
        if len(prior) != target_fold - 1 or len(prior) < 3:
            raise ValueError('The shortlist needs a full grid history for at least three prior folds.')
        grid = list(ParameterGrid(GRID_CLASSIFICATION))
        search_candidates = {}
        for expert in ('classic', 'hybrid'):
            totals = np.zeros(len(grid))
            for row in prior.values():
                candidates = row['selected'][expert]['round300']
                if row['grid_parameters'] != grid or sorted(idx for idx, _, _ in candidates) != list(range(len(grid))):
                    raise ValueError('Shortlist history must use the same full grid.')
                best = max(ap for _, ap, _ in candidates)
                for idx, ap, _ in candidates:
                    totals[idx] += best - ap
            search_candidates[expert] = sorted(range(len(grid)), key=lambda idx: (totals[idx], idx))[:3]
        name = f'fold{target_fold:02d}_aot_shortlist3'
        trial(name, dict(profiles['aot_prebin_router'], search_candidates=search_candidates), fold)
        if name not in records:
            raise RuntimeError(f'Fold failed: {name}')
        return pd.DataFrame([dict(fold=target_fold, seconds=records[name]['seconds'], **records[name]['scores'])])

    selected_profiles = os.environ.get('PQFM_SIH_HILL_PROFILES')
    if selected_profiles:
        names = [name.strip() for name in selected_profiles.split(',')]
        if os.environ.get('PQFM_SIH_AOT') and any(not profiles[name.removesuffix('_repeat')].get('aot') for name in names):
            raise ValueError('PQFM_SIH_AOT requires precompiled-only profiles.')
        for name in names:
            if name.endswith('_repeat'):
                trial(name, profiles[name.removesuffix('_repeat')])
            else:
                trial(name, profiles[name])
        missing = [name for name in names if name not in records]
        if missing:
            raise RuntimeError(f'Profiles failed: {missing}')
        return pd.DataFrame([dict(name=name, seconds=records[name]['seconds'], **records[name]['scores'])
                             for name in names]).sort_values('seconds')

    use_aot = bool(os.environ.get('PQFM_SIH_AOT'))
    profiles = {name: settings for name, settings in profiles.items() if bool(settings.get('aot')) == use_aot}
    if use_aot:
        for name, settings in profiles.items():
            trial(name, settings)
        if any(name not in records for name in profiles):
            raise RuntimeError('An AOT benchmark profile failed.')
        return pd.DataFrame([dict(name=name, seconds=records[name]['seconds'], **records[name]['scores'])
                             for name in profiles]).sort_values('seconds')
    # Front-load the newest full-grid and staged candidates when continuing a batch.
    if records:
        profiles = dict(sorted(profiles.items(), key=lambda entry: next((i for i, prefix in enumerate(('split_', 'fast_', 'early_', 'cuda_')) if entry[0].startswith(prefix)), 4)))
    # Full-sized first-fold runs share a warm process; no contention between trial profiles.
    for name, settings in profiles.items():
        trial(name, settings)
    complete = [records[name] for name in profiles if name in records]
    if len(complete) < 2:
        raise RuntimeError('Insufficient completed baseline profiles.')
    baseline = records.get('cpu16x4_hybrid', min(complete, key=lambda row: row['seconds']))

    def competitive(row):
        # Outer labels are reporting-only, including for staged search.
        return all(row['selected'][key]['validation_ap'] >= baseline['selected'][key]['validation_ap'] - 0.002
                   for key in ('classic', 'hybrid'))

    eligible = [row for row in complete if competitive(row) and not row['settings'].get('checkpoint')]
    best = min(eligible, key=lambda row: row['seconds'])
    for suffix, change in {
        'two_lanes': dict(lanes=2, workers=8, threads=3),
        'cpu_classic': dict(classic_gpus=[], classic_workers=16, workers=16, threads=3),
        'more_gpu': dict(workers=4, lanes=2, threads=4, inner=4),
        'sharded_shap': dict(shap_devices=[0, 1, 2, 3], resident=False),
    }.items():
        trial(best['name'] + '_' + suffix, dict(best['settings'], **change))
    exact = [row for row in records.values() if not row['settings'].get('checkpoint') and competitive(row)]
    best = min(exact, key=lambda row: row['seconds'])
    for checkpoint, promote in [(75, 6), (125, 9)]:
        trial(f'{best["name"]}_stage{checkpoint}', dict(best['settings'], checkpoint=checkpoint, promote=promote))
    contenders = sorted((row for row in records.values() if competitive(row)),
                        key=lambda row: row['seconds'])
    # Recheck one finalist per protocol using identical parameters and budgets.
    for staged in (False, True):
        finalist = next((row for row in contenders if bool(row['settings'].get('checkpoint')) == staged), None)
        if finalist:
            trial(finalist['name'] + '_repeat', finalist['settings'])
    return pd.DataFrame([dict(name=row['name'], seconds=row['seconds'], **row['scores'],
                              mean_process_cores=row['resources']['mean_process_cores'])
                         for row in records.values()]).sort_values('seconds')


if os.environ.get('PQFM_SIH_BUILD_WIDE'):
    compile_wide_encodings(os.environ['PQFM_SIH_BUILD_WIDE'])
elif os.environ.get('PQFM_SIH_FIRST_FOLD_HILL') == '1':
    display(first_fold_hillclimb())
    print('HILL_FINISHED', flush=True)


## Run and save each candidate

The soft score is `(1 − g) × classical + g × Heisenberg`, where `g` is the learned router probability. Models, Analysis predictions and per-row test predictions are saved alongside each fold result. The code resumes only results marked complete in this notebook’s separate output directory.

In [16]:
OUTPUT = Path(os.environ.get('PQFM_SIH_OUTPUT', '/home/pichau/pqfmlib-api/experiments/sih-sus/outputs/reference_soft_moe_sweep'))
OUTPUT.mkdir(parents=True, exist_ok=True)
import nbformat
import pqfmlib.experimental.heisenberg_tilelang as tilelang_backend
notebook_path = Path(os.environ.get('PQFM_SIH_NOTEBOOK', '/home/pichau/pqfmlib-api/experiments/sih-sus/sih_sus_reference_moe_sweep.ipynb'))
notebook_source = nbformat.read(notebook_path, as_version=4)
source_bytes = '\n'.join(c.source for c in notebook_source.cells if c.cell_type == 'code').encode()
metadata = {'notebook_sha256':hashlib.sha256(source_bytes).hexdigest(), 'dataset_sha256':dataset_sha256,
            'backend_sha256':hashlib.sha256(Path(tilelang_backend.__file__).read_bytes()).hexdigest(),
            'grid':GRID_CLASSIFICATION, 'sweep':SWEEP, 'protocol':'sih_6_plus_6_soft_moe_v1'}
metadata_path = OUTPUT / 'metadata.json'
PREVIOUS_COMPATIBLE_NOTEBOOK_SHA = '4c61f0a4e4f257d83a40437d0c8515ea225354bdb7aa3eb72fe4c127d0fda958'
if metadata_path.exists():
    existing = json.loads(metadata_path.read_text())
    history = existing.pop('implementation_history', [])
    if existing != metadata:
        same_experiment = all(existing.get(k) == metadata[k] for k in metadata if k != 'notebook_sha256')
        if not same_experiment or existing.get('notebook_sha256') != PREVIOUS_COMPATIBLE_NOTEBOOK_SHA:
            raise RuntimeError('Notebook, dataset, backend or configuration changed: use a new output directory.')
        history.append(dict(notebook_sha256=existing['notebook_sha256'], reason='Before fold-local inference reuse'))
    if history:
        metadata['implementation_history'] = history
metadata_path.write_text(json.dumps(metadata, indent=2))

def run_candidate(fold, name, config):
    if FOLD_CACHE.get('_fold') != fold['fold']:
        FOLD_CACHE.clear()
        FOLD_CACHE['_fold'] = fold['fold']
    folder = OUTPUT / f"fold_{fold['fold']:02d}" / name
    folder.mkdir(parents=True, exist_ok=True)
    result_path = folder / 'result.json'
    if result_path.exists():
        saved = json.loads(result_path.read_text())
        artifacts = ('predictions.parquet', 'analysis.parquet', 'validation.parquet', 'models.joblib')
        complete = (saved.get('status') == 'complete' and saved.get('fold') == fold['fold']
                    and saved.get('candidate') == name and saved.get('config') == config
                    and saved.get('dataset_sha256') == dataset_sha256
                    and all((folder / item).is_file() and (folder / item).stat().st_size > 0 for item in artifacts))
        if complete:
            print(f"Resuming: completed fold {fold['fold']} / {name}; skipping.", flush=True)
            return saved
        print(f"Incomplete checkpoint for fold {fold['fold']} / {name}; recomputing.", flush=True)
    train, test = data.loc[fold['train']], data.loc[fold['test']]
    started = time.perf_counter()
    bundle = fit_moe_bundle(train, protocol, 'classificacao', GRID_CLASSIFICATION, heisenberg_config=config)
    classic, _ = _cached_expert_output(bundle['classic'], test, protocol, stage='outer_test')
    secondary = bundle['models']['moe_Heisenberg']['secondary']
    quantum, _ = _cached_expert_output(secondary, test, protocol, stage='outer_test')
    columns = protocol['features_numericas'] + protocol['features_categoricas']
    if 'outer_test_router_x' not in FOLD_CACHE:
        FOLD_CACHE['outer_test_router_x'] = bundle['router_preprocessor'].transform(test[columns])
    router_x = FOLD_CACHE['outer_test_router_x']
    gate = bundle['models']['moe_Heisenberg']['router'].predict_proba(router_x)[:,1]
    if not np.isfinite(gate).all() or ((gate < 0) | (gate > 1)).any():
        raise ValueError('Invalid router probabilities')
    soft = (1-gate)*classic + gate*quantum
    predictions = test[['id_linha','mes','morte']].copy()
    predictions['classic'], predictions['heisenberg'], predictions['router'], predictions['soft_moe'] = classic, quantum, gate, soft
    for weight in (0.1, 0.25, 0.5):
        predictions[f'fixed_blend_{weight:g}'] = (1-weight)*classic + weight*quantum
    for gamma in GAMMAS:
        predictions[f'hard_gate_{gamma:g}'] = np.where(gate > gamma, quantum, classic)
    predictions.to_parquet(folder / 'predictions.parquet.tmp', index=False)
    (folder / 'predictions.parquet.tmp').replace(folder / 'predictions.parquet')
    bundle['analysis_predictions'].to_parquet(folder / 'analysis.parquet.tmp', index=False)
    (folder / 'analysis.parquet.tmp').replace(folder / 'analysis.parquet')
    bundle['validation_predictions'].to_parquet(folder / 'validation.parquet.tmp', index=False)
    (folder / 'validation.parquet.tmp').replace(folder / 'validation.parquet')
    joblib.dump(bundle, folder / 'models.joblib.tmp', compress=3)
    (folder / 'models.joblib.tmp').replace(folder / 'models.joblib')
    result = {'status':'complete','fold':fold['fold'],'month':str(fold['month'].date()),'candidate':name,'config':config,
              'n_train':len(train),'n_test':len(test),'roles':fold['roles'],'dataset_sha256':dataset_sha256,
              'notebook_sha256':metadata['notebook_sha256'],
              'classic_ap':float(average_precision_score(test.morte,classic)),
              'heisenberg_ap':float(average_precision_score(test.morte,quantum)),
              'soft_moe_ap':float(average_precision_score(test.morte,soft)),
              'fixed_blend_ap':{str(w):float(average_precision_score(test.morte,predictions[f'fixed_blend_{w:g}'])) for w in (0.1,0.25,0.5)},
              'hard_gate_ap':{str(g):float(average_precision_score(test.morte,predictions[f'hard_gate_{g:g}'])) for g in GAMMAS},
              'seconds':time.perf_counter()-started}
    temp = result_path.with_suffix('.tmp')
    temp.write_text(json.dumps(result,indent=2)); temp.replace(result_path)
    print(json.dumps(result), flush=True)
    return result

def run_sweep():
    requested_fold = os.environ.get('PQFM_SIH_FOLD')
    selected_folds = folds if requested_fold is None else [fold for fold in folds if fold['fold'] == int(requested_fold)]
    if not selected_folds:
        raise ValueError(f'Unknown fold: {requested_fold}')
    for fold in selected_folds:
        FOLD_CACHE.clear()
        # Consecutive SHAP-width candidates share an identical representation.
        order = ['reference', 'shap_20', 'shap_80'] + [name for name in SWEEP if name not in {'reference', 'shap_20', 'shap_80'}]
        if 'PQFM_SIH_CANDIDATES' in os.environ:
            order = os.environ['PQFM_SIH_CANDIDATES'].split(',')
        for name in order:
            config = SWEEP[name]
            print(f"Starting fold {fold['fold']}: {name}", flush=True)
            run_candidate(fold, name, config)


In [ ]:
if os.environ.get('PQFM_SIH_FIRST_FOLD_HILL') != '1':
    run_sweep()

Starting fold 1: reference
Resuming: completed fold 1 / reference; skipping.
Starting fold 1: shap_20
Resuming: completed fold 1 / sStarting fold 1: referencen]: 54/54 componentes; 0 posições de encoding preenchidas com zero pelo mapa.
Heisenberg [inner_train]: tilelang/naive_statevector/GPU; expectativas=exact; 19 qubits; mapa preparado, extração ainda não iniciada
PCA Heisenberg [expert]: 54/54 componentes; 0 posições de encoding preenchidas com zero pelo mapa.
Heisenberg [expert]: tilelang/naive_statevector/GPU; expectativas=exact; 19 qubits; mapa preparado, extração ainda não iniciada
2026-11-12 13:05:32  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:137): TileLang begins to compile kernel `kernel` with `out_idx=None`
2026-11-12 13:05:37  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:145): TileLang completes to compile kernel `kernel`
2026-11-12 13:05:37  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:137): TileLang begins to compile kernel `kernel` with `out_idx=None`
2026-11-12

/home/pichau/pqfmlib-api/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_data.py:1077: RuntimeWarning: invalid value encountered in sqrt
  xp.sqrt(self.var_), copy=False, constant_mask=constant_mask


2026-11-12 15:22:03  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:137): TileLang begins to compile kernel `kernel` with `out_idx=None`
2026-11-12 15:22:10  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:145): TileLang completes to compile kernel `kernel`
{"status": "complete", "fold": 2, "month": "2025-04-01", "candidate": "pca_16", "config": {"q_enc": 19, "n_components": 16, "R": 2, "alpha": 0.1, "hybrid_shap_top_k": 50}, "n_train": 174936, "n_test": 12483, "roles": [["2024-01-01", "2024-03-01", 33803], ["2024-04-01", "2024-09-01", 72259], ["2024-10-01", "2025-03-01", 68874]], "dataset_sha256": "395b5b355d4c227074e598f4224b0fb6cbb13ec55a4184ff2f2bae5f6ab0436b", "notebook_sha256": "d10434f732ea21a13d4bb76254f6cb389999a9d824a6e073304c1d49c06cc7fd", "classic_ap": 0.3093117737987814, "heisenberg_ap": 0.2876798863842767, "soft_moe_ap": 0.3081454008028569, "fixed_blend_ap": {"0.1": 0.30835064888149327, "0.25": 0.30677017208879187, "0.5": 0.3021458852577083}, "hard_gate_ap": {"0.05": 0.298

/home/pichau/pqfmlib-api/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_data.py:1077: RuntimeWarning: invalid value encountered in sqrt
  xp.sqrt(self.var_), copy=False, constant_mask=constant_mask


2026-11-12 15:29:49  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:137): TileLang begins to compile kernel `kernel` with `out_idx=None`
2026-11-12 15:29:56  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:145): TileLang completes to compile kernel `kernel`
{"status": "complete", "fold": 2, "month": "2025-04-01", "candidate": "pca_32", "config": {"q_enc": 19, "n_components": 32, "R": 2, "alpha": 0.1, "hybrid_shap_top_k": 50}, "n_train": 174936, "n_test": 12483, "roles": [["2024-01-01", "2024-03-01", 33803], ["2024-04-01", "2024-09-01", 72259], ["2024-10-01", "2025-03-01", 68874]], "dataset_sha256": "395b5b355d4c227074e598f4224b0fb6cbb13ec55a4184ff2f2bae5f6ab0436b", "notebook_sha256": "d10434f732ea21a13d4bb76254f6cb389999a9d824a6e073304c1d49c06cc7fd", "classic_ap": 0.3093117737987814, "heisenberg_ap": 0.2856765314776175, "soft_moe_ap": 0.3084282815358966, "fixed_blend_ap": {"0.1": 0.30831881928567473, "0.25": 0.3064395495985177, "0.5": 0.3017254293487883}, "hard_gate_ap": {"0.05": 0.2968

/home/pichau/pqfmlib-api/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_data.py:1077: RuntimeWarning: invalid value encountered in sqrt
  xp.sqrt(self.var_), copy=False, constant_mask=constant_mask
/home/pichau/pqfmlib-api/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_data.py:1077: RuntimeWarning: invalid value encountered in sqrt
  xp.sqrt(self.var_), copy=False, constant_mask=constant_mask


... (19 outputs left out)

/home/pichau/pqfmlib-api/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_data.py:1077: RuntimeWarning: invalid value encountered in sqrt
  xp.sqrt(self.var_), copy=False, constant_mask=constant_mask


2026-11-13 00:20:23  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:137): TileLang begins to compile kernel `kernel` with `out_idx=None`
2026-11-13 00:20:29  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:145): TileLang completes to compile kernel `kernel`
{"status": "complete", "fold": 7, "month": "2025-09-01", "candidate": "pca_16", "config": {"q_enc": 19, "n_components": 16, "R": 2, "alpha": 0.1, "hybrid_shap_top_k": 50}, "n_train": 237444, "n_test": 12531, "roles": [["2024-01-01", "2024-08-01", 94422], ["2024-09-01", "2025-02-01", 68676], ["2025-03-01", "2025-08-01", 74346]], "dataset_sha256": "395b5b355d4c227074e598f4224b0fb6cbb13ec55a4184ff2f2bae5f6ab0436b", "notebook_sha256": "d10434f732ea21a13d4bb76254f6cb389999a9d824a6e073304c1d49c06cc7fd", "classic_ap": 0.3270752112362789, "heisenberg_ap": 0.30748246837024107, "soft_moe_ap": 0.327272545840154, "fixed_blend_ap": {"0.1": 0.32658097097398925, "0.25": 0.326469364905705, "0.5": 0.32241544333387956}, "hard_gate_ap": {"0.05": 0.3267

/home/pichau/pqfmlib-api/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_data.py:1077: RuntimeWarning: invalid value encountered in sqrt
  xp.sqrt(self.var_), copy=False, constant_mask=constant_mask
/home/pichau/pqfmlib-api/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_data.py:1077: RuntimeWarning: invalid value encountered in sqrt
  xp.sqrt(self.var_), copy=False, constant_mask=constant_mask


2026-11-13 00:34:21  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:137): TileLang begins to compile kernel `kernel` with `out_idx=None`
2026-11-13 00:34:27  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:145): TileLang completes to compile kernel `kernel`
Router: tuned; learning_rate=0.2; refit=74346 amostras.
PCA Heisenberg [outer_refit]: 32/32 componentes; 0 posições de encoding preenchidas com zero pelo mapa.
Heisenberg [outer_refit]: tilelang/naive_statevector/GPU; expectativas=exact; 19 qubits; mapa preparado, extração ainda não iniciada
2026-11-13 00:35:04  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:137): TileLang begins to compile kernel `kernel` with `out_idx=None`
2026-11-13 00:35:09  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:145): TileLang completes to compile kernel `kernel`
2026-11-13 00:36:40  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:137): TileLang begins to compile kernel `kernel` with `out_idx=None`
2026-11-13 00:36:46  [TileLang:tilelang.jit.kernel:INFO]